> **If this notebook fails:** Run `python run_pipeline.py` in the terminal instead. It does the same thing and writes output to `pipeline_output/`.

# Anomaly Detection Pipeline

Multi-dimensional sensor data: **8 sensors × 10 heating steps × m GR values** per block.

**Pipeline:**
1. Calibration (Normal Air per substance)
2. Normalization
3. Sensor validity check
4. Feature extraction
5. Visualization — statistics (80 raw + 80 normalized plots per substance)
6. PCA / UMAP / t-SNE

In [89]:
import sys
from pathlib import Path

# Add project folder to path
_here = Path.cwd()
if (_here / "data_utils.py").exists():
    sys.path.insert(0, str(_here))
else:
    _alt = _here / "Nw File Anomaly Detection"
    if (_alt / "data_utils.py").exists():
        sys.path.insert(0, str(_alt))

import json
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from data_utils import (
    BASE_DIR, SUBSTANCES, N_SENSORS, N_STEPS, STAT_NAMES, N_WARMUP, M_STEADY,
    TARGET_SAMPLES_PER_SUBSTANCE, WINDOW_STRIDE,
    load_folder_blocks, compute_baseline, compute_stats, apply_windowing,
    apply_windowing_multi, normalize_block, extract_features, get_valid_sensors,
)

SUBSTANCE_PREFIX = {"Acetone": "Acetone", "Redidlo": "Redidlo", "Softasept": "Softasept", "Savo": "Savo", "Vinegar": "Vinegar"}
print("BASE_DIR:", BASE_DIR)

BASE_DIR: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection


## Step 1: Calibration (Normal Air per substance)

Collect p blocks of Normal Air. For each sensor×HS, compute median, std, etc.

In [90]:
calibration = {}
normal_blocks = {}

for sub in SUBSTANCES:
    folder = BASE_DIR / sub
    if not folder.exists():
        print(f"Skip {sub}: folder not found")
        continue
    # Try Normal_Air_* (consolidated) or Normal* (legacy)
    blocks, labels = load_folder_blocks(folder, "Normal_Air_*")
    if not blocks:
        blocks, labels = load_folder_blocks(folder, "Normal*")
    if not blocks:
        blocks, labels = load_folder_blocks(folder, "Nomal*")
    windowed = [apply_windowing(b, n=N_WARMUP, m=M_STEADY) for b in blocks]
    baseline = compute_baseline(windowed, apply_windowing_flag=False)
    calibration[sub] = baseline
    normal_blocks[sub] = (windowed, labels)
    print(f"{sub}: {len(blocks)} Normal Air blocks, baseline for {len(baseline)} (sensor,step) pairs")

Acetone: 22 Normal Air blocks, baseline for 80 (sensor,step) pairs
Redidlo: 26 Normal Air blocks, baseline for 80 (sensor,step) pairs
Softasept: 24 Normal Air blocks, baseline for 80 (sensor,step) pairs
Savo: 24 Normal Air blocks, baseline for 80 (sensor,step) pairs
Vinegar: 26 Normal Air blocks, baseline for 80 (sensor,step) pairs


## Step 2 & 3: Normalization + Sensor validity

Normalize raw GR using baseline. Exclude invalid sensors per block.

In [91]:
def load_substance_data(sub: str):
    """Return (norm_blocks, norm_labels, sub_blocks, sub_labels). Uses sliding windows for ~300 samples per substance."""
    folder = BASE_DIR / sub
    prefix = SUBSTANCE_PREFIX.get(sub, sub)
    norm_blks, norm_lbls = normal_blocks.get(sub, ([], []))
    sub_blocks, sub_labels = load_folder_blocks(folder, f"{prefix}_*")
    sub_windowed, sub_windowed_labels = [], []
    for b, lbl in zip(sub_blocks, sub_labels):
        for i, w in enumerate(apply_windowing_multi(b, stride=WINDOW_STRIDE)):
            sub_windowed.append(w)
            sub_windowed_labels.append(f"{lbl}_w{i}" if i > 0 else lbl)
    return norm_blks, norm_lbls, sub_windowed, sub_windowed_labels


def process_blocks(blocks, labels, baseline, substance_label, return_blockwise=False):
    """Normalize blocks, apply validity check, return raw/norm data."""
    raw_by_sensor_step = defaultdict(list)
    norm_by_sensor_step = defaultdict(list)
    valid_blocks = []
    raw_blocks = []
    norm_blocks = []
    for blk, lbl in zip(blocks, labels):
        valid_s = get_valid_sensors(blk)
        if len(valid_s) < 4:
            continue
        norm_blk = normalize_block(blk, baseline)
        raw_blocks.append(blk)
        norm_blocks.append(norm_blk)
        valid_blocks.append({
            "label": lbl,
            "valid_sensors": valid_s,
        })
        for (s, st), vals in blk.items():
            if s in valid_s and vals:
                raw_by_sensor_step[(s, st)].extend(vals)
        for (s, st), arr in norm_blk.items():
            if s in valid_s and len(arr) > 0:
                norm_by_sensor_step[(s, st)].extend(arr.tolist())
    if return_blockwise:
        return raw_blocks, norm_blocks, valid_blocks
    return raw_by_sensor_step, norm_by_sensor_step, valid_blocks

In [92]:
# process_blocks is defined in the cell above

## Step 4: Feature extraction

In [93]:
all_features_raw = []
all_features_calib = []
all_labels = []  # substance name
rng = np.random.default_rng(42)

for sub in SUBSTANCES:
    if sub not in calibration:
        continue
    norm_blks, norm_lbls, sub_blks, sub_lbls = load_substance_data(sub)
    baseline = calibration[sub]
    for blocks, labels, tag in [(norm_blks, norm_lbls, "Normal_Air"), (sub_blks, sub_lbls, sub)]:
        if len(labels) != len(blocks):
            labels = [f"b{i}" for i in range(len(blocks))]
        raw_r, norm_r, valid_blocks = process_blocks(blocks, labels, baseline, tag, return_blockwise=True)
        indices = list(range(len(valid_blocks)))
        if tag != "Normal_Air" and len(indices) > TARGET_SAMPLES_PER_SUBSTANCE:
            indices = rng.choice(len(valid_blocks), TARGET_SAMPLES_PER_SUBSTANCE, replace=False)
        for i in indices:
            sample = valid_blocks[i]
            raw_blk = raw_r[i]
            calib_blk = norm_r[i]
            valid_s = sample["valid_sensors"]
            all_features_raw.append(extract_features(raw_blk, baseline, valid_s))
            all_features_calib.append(extract_features(calib_blk, baseline, valid_s))
            all_labels.append(tag)

X_raw = np.vstack(all_features_raw)
X_calib = np.vstack(all_features_calib)
X = X_calib  # default alias for downstream cells
y = np.array(all_labels)
print(f"X_raw: {X_raw.shape}, X_calib: {X_calib.shape}, labels: {len(np.unique(y))} classes")
for lab in np.unique(y):
    print(f"  {lab}: {(y == lab).sum()} samples")

X_raw: (1620, 800), X_calib: (1620, 800), labels: 6 classes
  Acetone: 300 samples
  Normal_Air: 120 samples
  Redidlo: 300 samples
  Savo: 300 samples
  Softasept: 300 samples
  Vinegar: 300 samples


## Step 5: Visualization — 80 plots raw + 80 normalized per substance

80 = 8 sensors × 10 HS. Each subplot shows distribution of values.

In [94]:
OUTPUT_DIR = BASE_DIR / "pipeline_output"
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "stats_plots").mkdir(exist_ok=True)

def plot_80_grid(data_by_sensor_step, title, filepath, stat_key="median"):
    """80 subplots: 8 sensors × 10 HS. Show distribution (e.g. median of values) per (sensor, step)."""
    fig, axes = plt.subplots(8, 10, figsize=(18, 12))
    for s in range(N_SENSORS):
        for st in range(N_STEPS):
            ax = axes[s, st]
            vals = data_by_sensor_step.get((s, st), [])
            if vals:
                arr = np.array(vals)
                stats = compute_stats(arr)
                v = stats.get(stat_key, 0)
                if np.isnan(v) or np.isinf(v):
                    v = 0
                ax.bar([stat_key], [float(v)], color="steelblue", edgecolor="black")
            ax.set_xticks([])
            ax.set_title(f"S{s} HS{st}", fontsize=7)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(filepath, dpi=100, bbox_inches="tight")
    plt.close()


for sub in SUBSTANCES:
    if sub not in calibration:
        continue
    norm_blks, norm_lbls, sub_blks, sub_lbls = load_substance_data(sub)
    baseline = calibration[sub]
    raw_r, norm_r, _ = process_blocks(norm_blks + sub_blks, [""] * (len(norm_blks) + len(sub_blks)), baseline, sub)
    plot_80_grid(raw_r, f"{sub} — Raw GR (median)", OUTPUT_DIR / f"stats_plots/{sub}_raw_median.png")
    plot_80_grid(norm_r, f"{sub} — Normalized GR (median)", OUTPUT_DIR / f"stats_plots/{sub}_normalized_median.png")
    print(f"Saved plots for {sub}")

Saved plots for Acetone
Saved plots for Redidlo
Saved plots for Softasept
Saved plots for Savo
Saved plots for Vinegar


### Step 5 (alt): 80 plots as histogram/distribution per (sensor, HS)

In [95]:
def plot_80_histograms(data_by_sensor_step, title, filepath, bins=30):
    fig, axes = plt.subplots(8, 10, figsize=(20, 14))
    for s in range(N_SENSORS):
        for st in range(N_STEPS):
            ax = axes[s, st]
            vals = data_by_sensor_step.get((s, st), [])
            if vals:
                ax.hist(vals, bins=bins, color="steelblue", edgecolor="white", alpha=0.8)
            ax.set_title(f"S{s} HS{st}", fontsize=7)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(filepath, dpi=100, bbox_inches="tight")
    plt.close()


for sub in SUBSTANCES:
    if sub not in calibration:
        continue
    norm_blks, norm_lbls, sub_blks, sub_lbls = load_substance_data(sub)
    baseline = calibration[sub]
    raw_r, norm_r, _ = process_blocks(norm_blks + sub_blks, [""] * (len(norm_blks) + len(sub_blks)), baseline, sub)
    plot_80_histograms(raw_r, f"{sub} — Raw GR distribution", OUTPUT_DIR / f"stats_plots/{sub}_raw_hist.png")
    plot_80_histograms(norm_r, f"{sub} — Normalized GR distribution", OUTPUT_DIR / f"stats_plots/{sub}_norm_hist.png")
    print(f"Saved histograms for {sub}")

Saved histograms for Acetone
Saved histograms for Redidlo
Saved histograms for Softasept
Saved histograms for Savo
Saved histograms for Vinegar


## Step 6: PCA / UMAP / t-SNE

In [96]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

# Switch between raw and normalized features for dim reduction
use_calibrated = True  # Set to False to use raw features (not normalized by baseline)
X_selected = X_calib if use_calibrated else X_raw

# Fill NaN with 0 for decomposition
X_clean = np.nan_to_num(X_selected, nan=0.0, posinf=0.0, neginf=0.0)
X_scaled = StandardScaler().fit_transform(X_clean)

# Exclude Normal Air from dim reduction — fit and transform on substances only
sub_mask = y != "Normal_Air"
X_sub = X_scaled[sub_mask]
y_sub = y[sub_mask]
print(f"Dim reduction: {len(X_sub)} substance blocks (Normal Air excluded)")

# PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sub)
print(f"PCA: explained variance {pca.explained_variance_ratio_.sum():.2%}")

# t-SNE (perplexity must be 5-50 and < n_samples)
perplexity = min(30, max(5, len(X_sub) - 1))
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
X_tsne = tsne.fit_transform(X_sub)

# UMAP (if available)
if HAS_UMAP:
    reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X_sub)
else:
    X_umap = X_pca  # fallback

Dim reduction: 1500 substance blocks (Normal Air excluded)
PCA: explained variance 86.12%


In [97]:
# Plot dim reduction (substances only; embeddings already exclude Normal Air)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, emb, name in zip(axes, [X_pca, X_tsne, X_umap], ["PCA", "t-SNE", "UMAP"]):
    for lab in np.unique(y_sub):
        mask = y_sub == lab
        ax.scatter(emb[mask, 0], emb[mask, 1], label=lab, alpha=0.6, s=40)
    ax.set_title(name)
    ax.legend(loc="best", fontsize=8)
plt.suptitle("Step 6: Dimensionality reduction (substances only)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dim_reduction.png", dpi=120, bbox_inches="tight")
plt.show()

## Step 11: Anomaly scoring

Model trained on Normal Air, evaluated on all substances.

In [98]:
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

anomaly_train_label = "Vinegar"
normal_mask = y == anomaly_train_label
X_normal = X_scaled[normal_mask]
X_all = X_scaled

n_components_iforest = min(60, X_normal.shape[0], X_normal.shape[1])
if n_components_iforest < 1:
    raise ValueError("Need at least one training sample for anomaly PCA/IsolationForest.")

anomaly_pca = PCA(n_components=n_components_iforest, random_state=42)
X_normal_reduced = anomaly_pca.fit_transform(X_normal)
X_all_reduced = anomaly_pca.transform(X_all)
print(f"IsolationForest PCA: {X_all.shape[1]} -> {n_components_iforest}, explained variance {anomaly_pca.explained_variance_ratio_.sum():.2%}")

model = IsolationForest(contamination=0.1, random_state=42)
model.fit(X_normal_reduced)
scores = model.decision_function(X_all_reduced)
preds = model.predict(X_all_reduced)

print(f"Anomaly results (trained on {anomaly_train_label}):")
for lab in np.unique(y):
    mask = y == lab
    n_anom = (preds[mask] == -1).sum()
    print(f"  {lab}: {mask.sum()} blocks, {n_anom} anomalies ({100*n_anom/mask.sum():.1f}%)")

pd.DataFrame({"label": y, "score": scores, "pred": preds}).to_csv(OUTPUT_DIR / "anomaly_scores.csv", index=False)

IsolationForest PCA: 800 -> 60, explained variance 99.83%
Anomaly results (trained on Vinegar):
  Acetone: 300 blocks, 255 anomalies (85.0%)
  Normal_Air: 120 blocks, 97 anomalies (80.8%)
  Redidlo: 300 blocks, 146 anomalies (48.7%)
  Savo: 300 blocks, 208 anomalies (69.3%)
  Softasept: 300 blocks, 298 anomalies (99.3%)
  Vinegar: 300 blocks, 30 anomalies (10.0%)


# TODO
## You have done windowing, 800 stats feature extraction - good foundation, but...
## I reduced feature dimensions from 800 to 60 -> IsolationForest has significantly better performance, why?

## Please do next week: 
### Study in depth the problem of training data size->input feature size->model complexity, otherwise all your models will be probably overfitted! Please propose chapter and some tables in your dimploma thesis doc with these informations. 
### Try to propose Dense autoencoder architecture with respect to "Curse of Dimensionality" problem, do not exceed the model recommended number of total parameters (about 1000-2000 parameters? Check it)!
### Study and apply semi or full automatic approaches how to extract features from windowed time serie signal - for example using of 1D convolution
### So you will have methods:
#### 1. Time serie->windowed data->manual feature extraction (800 stats)->feature selection (dimension reduction - PCA or other methods)
#### 2. Time serie->windowed data->automatic feature extraction (XXX features)->feature selection (dimension reduction - PCA or other methods)

### You will have a models:
#### 1. Isolation Forest with proper tuned hyperparameters (based on "Curse of Dimensionality" problem)
#### 2. Dense Autoencoder with proper tuned hyperparameters (based on "Curse of Dimensionality" problem)
#### 3. finally, you can add the LSTM autoencoder approach, but probably you have to study what are the best (proper) bme688 data representation for such model: raw timeseries in proper time order without feature extraction?


## Final experiments
### I will take for you new batch of "Normal Air" data next week, every day in server room.
### You will take one week Normal Air as training data and analyse it with proposed models:
#### How anomaly detectors work on substance anomaly data from lab? Is anomaly detection sucessfull?
#### How anomaly detectors work on partial Normal Air data which detector see for the first time? For example detector will be trained on Normal data from Monday-Wednesday, will classify the Normal data from Thursday-Friday also as Normal Air? Do not forget if you will use calibrated substance data in final evaluation, you should calibrate also one-week Normal Air data - based on selected interval (for example Monday measurements) you will compute the same statistics as you did in substance timeseries calibration. The valid approach can be to implement posibility to switch between calibrated and noncalibrated data to see differences.
#### These will be the key experiments. To be able say how stable is your model for anomaly detection during the time, what thresholds are set for anomaly decision and after what period probably the model should be retrained with new Normal Air data.


> **Deprecated — do not run for submission.**  
> Use **Step 12-Final (submission pipeline)** at the end of this notebook for the unified pipeline (Steps 12–17) and final outputs under pipeline_output/final_clean_run/.

---

## Step 12: Week-Normal-Air TODO completion

This section completes the TODO experiments for **next-week Normal Air data**:

1. Convert week Normal Air files to a consolidated CSV.
2. Train detectors on Monday-Wednesday Normal Air.
3. Test on Thursday-Friday Normal Air (detector should still classify as normal).
4. Evaluate anomaly detection on substance data from lab.
5. Compare **calibrated vs non-calibrated** variants.
6. Save summary tables to `pipeline_output/`.

In [99]:
from data_utils import load_blocks_from_file
import re
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor


def _resolve_week_normal_folder(base_dir: Path) -> Path | None:
    # Priority: exact folder requested by user
    candidates = [
        "NormalAir",
        "Normal Air Week", "Normal_Air_Week", "Week Normal Air", "Air Week", "Air New"
    ]
    for c in candidates:
        p = base_dir / c
        if p.exists() and p.is_dir():
            return p
    return None


def _infer_day_tag(path_like: str) -> str:
    """Infer weekday from full path (folder names like 04_MON, 01_WED, etc.) and filename fallback."""
    s = str(path_like).lower()
    mapping = {
        "mon": "Monday", "monday": "Monday",
        "tue": "Tuesday", "tues": "Tuesday", "tuesday": "Tuesday",
        "wed": "Wednesday", "wednesday": "Wednesday",
        "thu": "Thursday", "thur": "Thursday", "thurs": "Thursday", "thursday": "Thursday",
        "fri": "Friday", "friday": "Friday",
        "sat": "Saturday", "saturday": "Saturday",
        "sun": "Sunday", "sunday": "Sunday",
    }
    for k, v in mapping.items():
        if k in s:
            return v

    # fallback by number token 1..7 -> Mon..Sun
    m = re.search(r"(?:^|[^0-9])(\d{1,2})(?:[^0-9]|$)", s)
    if m:
        idx = int(m.group(1))
        week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
        if 1 <= idx <= 7:
            return week[idx - 1]
    return "Unknown"


def export_week_normal_to_csv(base_dir: Path, output_csv: Path):
    week_folder = _resolve_week_normal_folder(base_dir)
    if week_folder is None:
        print("[WARN] Week Normal Air folder not found. Expected e.g. 'Normal Air Week' / 'Air New'.")
        return None

    # Recursive scan to include weekday subfolders (e.g. 04_MON, 01_WED, ...)
    files = sorted([f for f in week_folder.rglob("*") if f.is_file() and f.suffix.lower() in {".bmerawdata", ".csv"}])
    if not files:
        print(f"[WARN] No .bmerawdata/.csv files found in {week_folder}")
        return None

    rows = []
    for f in files:
        day = _infer_day_tag(f)
        blocks = load_blocks_from_file(f)
        for b_idx, blk in enumerate(blocks):
            for (s, st), vals in blk.items():
                for i, v in enumerate(vals):
                    rows.append({
                        "source_file": f.name,
                        "day": day,
                        "block_id": b_idx,
                        "sensor": s,
                        "step": st,
                        "sample_idx": i,
                        "gr": float(v),
                    })

    if not rows:
        print("[WARN] No parsed values from week normal files.")
        return None

    df_week = pd.DataFrame(rows)
    output_csv.parent.mkdir(exist_ok=True, parents=True)
    df_week.to_csv(output_csv, index=False)
    print(f"Saved week-normal long CSV: {output_csv}")
    print(df_week.groupby('day').size().sort_index())
    return df_week


week_csv_path = OUTPUT_DIR / "week_normal_air_long.csv"
df_week_long = export_week_normal_to_csv(BASE_DIR, week_csv_path)

Saved week-normal long CSV: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\week_normal_air_long.csv
day
Friday       102997
Monday       107395
Thursday     123283
Tuesday      137390
Wednesday    125550
dtype: int64


In [100]:
def _build_week_feature_dataset(base_dir: Path, calibration_dict: dict, calibrated: bool = True):
    week_folder = _resolve_week_normal_folder(base_dir)
    if week_folder is None:
        return None, None, None

    # Recursive scan to include weekday subfolders (e.g. 04_MON, 05_TUE, ...)
    files = sorted([f for f in week_folder.rglob("*") if f.is_file() and f.suffix.lower() in {".bmerawdata", ".csv"}])
    X_week, day_week, sample_name = [], [], []

    # Build week-normal calibration from Monday files (as requested in TODO),
    # fallback to first available baseline if Monday data are missing.
    monday_blocks = []
    for f in files:
        if _infer_day_tag(f) == "Monday":
            for blk in load_blocks_from_file(f):
                monday_blocks.append(apply_windowing(blk, n=N_WARMUP, m=M_STEADY))

    if monday_blocks:
        baseline_ref = compute_baseline(monday_blocks, apply_windowing_flag=False)
    else:
        baseline_ref = calibration_dict.get("Acetone") if "Acetone" in calibration_dict else next(iter(calibration_dict.values()), None)

    if baseline_ref is None:
        print("[WARN] No calibration baseline available. Run Step 1 first.")
        return None, None, None

    for f in files:
        day = _infer_day_tag(f)
        blocks = load_blocks_from_file(f)
        for b_i, blk in enumerate(blocks):
            w_blk = apply_windowing(blk, n=N_WARMUP, m=M_STEADY)
            valid_s = get_valid_sensors(w_blk)
            if len(valid_s) < 4:
                continue
            if calibrated:
                used_blk = normalize_block(w_blk, baseline_ref)
            else:
                used_blk = w_blk
            feats = extract_features(used_blk, baseline_ref, valid_s)
            X_week.append(feats)
            day_week.append(day)
            sample_name.append(f"{f.stem}_b{b_i}")

    if not X_week:
        return None, None, None
    return np.vstack(X_week), np.array(day_week), np.array(sample_name)


def _dense_autoencoder_param_count(input_dim: int, hidden=(12, 6, 12)) -> int:
    layers = [input_dim] + list(hidden) + [input_dim]
    total = 0
    for i in range(len(layers) - 1):
        total += layers[i] * layers[i + 1] + layers[i + 1]
    return total


def run_week_normal_experiments(calibrated: bool = True, tag: str = "calibrated"):
    X_week, days, names = _build_week_feature_dataset(BASE_DIR, calibration, calibrated=calibrated)
    if X_week is None:
        print(f"[WARN] Cannot run {tag} experiment: no week-normal dataset.")
        return None

    train_days = {"Monday", "Tuesday", "Wednesday"}
    test_days = {"Thursday", "Friday"}
    tr_mask = np.array([d in train_days for d in days])
    te_mask = np.array([d in test_days for d in days])

    if tr_mask.sum() < 10 or te_mask.sum() < 3:
        print(f"[WARN] Not enough week-normal samples for {tag}: train={tr_mask.sum()}, test={te_mask.sum()}")
        return None

    X_train_week = X_week[tr_mask]
    X_test_week = X_week[te_mask]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_week)
    X_test_scaled = scaler.transform(X_test_week)

    # Substance anomaly set from current notebook features
    normal_label = "Normal_Air"
    sub_mask = y != normal_label
    X_substance = X_calib[sub_mask] if calibrated else X_raw[sub_mask]
    X_sub_scaled = scaler.transform(X_substance)

    # --- Isolation Forest ---
    X_tr_fit, X_tr_cal = train_test_split(X_train_scaled, test_size=0.25, random_state=42)
    if_model = IsolationForest(n_estimators=300, contamination=0.05, random_state=42)
    if_model.fit(X_tr_fit)
    cal_scores = if_model.decision_function(X_tr_cal)
    thr_if = float(np.percentile(cal_scores, 5))

    pred_test_normal_if = (if_model.decision_function(X_test_scaled) < thr_if).astype(int)
    pred_sub_if = (if_model.decision_function(X_sub_scaled) < thr_if).astype(int)

    # --- Dense Autoencoder (small model) ---
    hidden = (12, 6, 12)
    n_params = _dense_autoencoder_param_count(X_train_scaled.shape[1], hidden=hidden)
    mlp = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation="relu",
        solver="adam",
        alpha=1e-3,
        max_iter=700,
        random_state=42,
    )
    mlp.fit(X_tr_fit, X_tr_fit)

    recon_cal = mlp.predict(X_tr_cal)
    err_cal = np.mean((X_tr_cal - recon_cal) ** 2, axis=1)
    thr_ae = float(np.percentile(err_cal, 95))

    recon_test = mlp.predict(X_test_scaled)
    err_test = np.mean((X_test_scaled - recon_test) ** 2, axis=1)
    pred_test_normal_ae = (err_test >= thr_ae).astype(int)

    recon_sub = mlp.predict(X_sub_scaled)
    err_sub = np.mean((X_sub_scaled - recon_sub) ** 2, axis=1)
    pred_sub_ae = (err_sub >= thr_ae).astype(int)

    # y_true: week Thu/Fri should be normal (0), substance should be anomaly (1)
    y_true_norm = np.zeros(len(pred_test_normal_if), dtype=int)
    y_true_sub = np.ones(len(pred_sub_if), dtype=int)

    # metrics tables
    rows = [
        {
            "setup": tag,
            "model": "IsolationForest",
            "train_days": "Mon-Wed",
            "test_days": "Thu-Fri",
            "week_normal_test_samples": int(len(pred_test_normal_if)),
            "week_normal_flagged_as_anomaly": int(pred_test_normal_if.sum()),
            "week_normal_correct_rate": float((pred_test_normal_if == y_true_norm).mean()),
            "substance_samples": int(len(pred_sub_if)),
            "substance_detected_as_anomaly": int(pred_sub_if.sum()),
            "substance_detection_rate": float((pred_sub_if == y_true_sub).mean()),
            "threshold": thr_if,
            "model_params": None,
        },
        {
            "setup": tag,
            "model": "DenseAutoencoder(MLP)",
            "train_days": "Mon-Wed",
            "test_days": "Thu-Fri",
            "week_normal_test_samples": int(len(pred_test_normal_ae)),
            "week_normal_flagged_as_anomaly": int(pred_test_normal_ae.sum()),
            "week_normal_correct_rate": float((pred_test_normal_ae == y_true_norm).mean()),
            "substance_samples": int(len(pred_sub_ae)),
            "substance_detected_as_anomaly": int(pred_sub_ae.sum()),
            "substance_detection_rate": float((pred_sub_ae == y_true_sub).mean()),
            "threshold": thr_ae,
            "model_params": int(n_params),
        },
    ]

    print(f"\n[{tag}] Week-normal split: train={tr_mask.sum()} (Mon-Wed), test={te_mask.sum()} (Thu-Fri)")
    print(f"[{tag}] Dense AE parameter count: {n_params}")
    if 1000 <= n_params <= 2000:
        print(f"[{tag}] Dense AE params are in recommended range (1000-2000).")
    else:
        print(f"[{tag}] Dense AE params are OUTSIDE recommended range (1000-2000).")

    return pd.DataFrame(rows)


res_cal = run_week_normal_experiments(calibrated=True, tag="calibrated")
res_raw = run_week_normal_experiments(calibrated=False, tag="non_calibrated")

summary_frames = [d for d in [res_cal, res_raw] if d is not None]
if summary_frames:
    summary_df = pd.concat(summary_frames, ignore_index=True)
    summary_path = OUTPUT_DIR / "week_normal_experiment_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"\nSaved summary: {summary_path}")
    display(summary_df)
else:
    print("No week-normal experiment results generated.")


[calibrated] Week-normal split: train=15 (Mon-Wed), test=9 (Thu-Fri)
[calibrated] Dense AE parameter count: 20174
[calibrated] Dense AE params are OUTSIDE recommended range (1000-2000).

[non_calibrated] Week-normal split: train=15 (Mon-Wed), test=9 (Thu-Fri)
[non_calibrated] Dense AE parameter count: 20174
[non_calibrated] Dense AE params are OUTSIDE recommended range (1000-2000).

Saved summary: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\week_normal_experiment_summary.csv


,setup,model,train_days,test_days,week_normal_test_samples,week_normal_flagged_as_anomaly,week_normal_correct_rate,substance_samples,substance_detected_as_anomaly,substance_detection_rate,threshold,model_params
0,calibrated,IsolationForest,Mon-Wed,Thu-Fri,9,2,0.777778,1500,699,0.466000,0.067814,NaN
1,calibrated,DenseAutoencoder(MLP),Mon-Wed,Thu-Fri,9,0,1.000000,1500,118,0.078667,0.872995,20174.0
2,non_calibrated,IsolationForest,Mon-Wed,Thu-Fri,9,2,0.777778,1500,902,0.601333,0.070145,NaN
3,non_calibrated,DenseAutoencoder(MLP),Mon-Wed,Thu-Fri,9,1,0.888889,1500,1050,0.700000,0.315524,20174.0


> **Deprecated — do not run for submission.**  
> Use **Step 12-Final (submission pipeline)** at the end of this notebook for the unified pipeline (Steps 12–17) and final outputs under pipeline_output/final_clean_run/.

---

## Step 13: Advanced TODO completion (threshold sweep, compact AE, auto features, stability)

This section completes the remaining TODO items:

1. Threshold sweep for IF and AE (FPR vs detection tradeoff).
2. Dense AE complexity reduction to ~1000–2000 parameters.
3. Automatic feature extraction branch using simple 1D convolution features.
4. Stability-over-time analysis: train Mon-Wed, test Thu and Fri separately.
5. Final comparison/report table + retraining recommendation rule.

In [101]:
from sklearn.decomposition import PCA
from sklearn.metrics import auc


def _load_week_blocks_with_days(base_dir: Path):
    week_folder = _resolve_week_normal_folder(base_dir)
    if week_folder is None:
        print("[WARN] Week folder not found.")
        return [], [], []

    files = sorted([f for f in week_folder.rglob("*") if f.is_file() and f.suffix.lower() in {".bmerawdata", ".csv"}])
    blocks, days, names = [], [], []
    for f in files:
        d = _infer_day_tag(f)
        for i, blk in enumerate(load_blocks_from_file(f)):
            w_blk = apply_windowing(blk, n=N_WARMUP, m=M_STEADY)
            blocks.append(w_blk)
            days.append(d)
            names.append(f"{f.stem}_b{i}")
    return blocks, np.array(days), np.array(names)


def _compute_week_monday_baseline(week_blocks, week_days):
    mon_blks = [b for b, d in zip(week_blocks, week_days) if d == "Monday"]
    if len(mon_blks) == 0:
        return None
    return compute_baseline(mon_blks, apply_windowing_flag=False)


def _extract_conv_features(block: dict, valid_sensors=None):
    """Simple automatic features from 1D conv responses per (sensor,step)."""
    kernels = [np.array([1, -1], dtype=float), np.array([1, -2, 1], dtype=float), np.array([1, 0, -1], dtype=float)]
    if valid_sensors is None:
        valid_sensors = list(range(N_SENSORS))
    feats = []
    for s in range(N_SENSORS):
        for st in range(N_STEPS):
            if s not in valid_sensors:
                feats.extend([0.0, 0.0, 0.0])
                continue
            arr = np.array(block.get((s, st), []), dtype=float)
            if len(arr) < 3:
                feats.extend([0.0, 0.0, 0.0])
                continue
            vals = []
            for k in kernels:
                conv = np.convolve(arr, k, mode="valid")
                vals.append(float(np.mean(np.abs(conv))))
            feats.extend(vals)
    return np.array(feats, dtype=float)


def _build_substance_blocks(calibrated=True):
    sub_blocks = []
    sub_labels = []
    for sub in SUBSTANCES:
        if sub not in calibration:
            continue
        baseline = calibration[sub]
        _, _, sblks, slbls = load_substance_data(sub)
        labels = slbls if len(slbls) == len(sblks) else [f"b{i}" for i in range(len(sblks))]
        raw_blks, norm_blks, valid_meta = process_blocks(sblks, labels, baseline, sub, return_blockwise=True)
        for i in range(len(valid_meta)):
            use_blk = norm_blks[i] if calibrated else raw_blks[i]
            sub_blocks.append(use_blk)
            sub_labels.append(sub)
    return sub_blocks, np.array(sub_labels)


def _dense_param_count(input_dim: int, hidden=(16, 8, 16)):
    layers = [input_dim] + list(hidden) + [input_dim]
    total = 0
    for i in range(len(layers) - 1):
        total += layers[i] * layers[i+1] + layers[i+1]
    return total

In [102]:
# Build week-normal datasets and labels
week_blocks_raw, week_days, week_names = _load_week_blocks_with_days(BASE_DIR)
if len(week_blocks_raw) == 0:
    raise ValueError("No week-normal blocks found in NormalAir folder.")

baseline_week = _compute_week_monday_baseline(week_blocks_raw, week_days)
if baseline_week is None:
    raise ValueError("No Monday blocks found for week-normal baseline calibration.")

# Keep only blocks with enough valid sensors
wk_raw_kept, wk_cal_kept, wk_days_kept, wk_valid_kept = [], [], [], []
for b, d in zip(week_blocks_raw, week_days):
    valid_s = get_valid_sensors(b)
    if len(valid_s) < 4:
        continue
    wk_raw_kept.append(b)
    wk_cal_kept.append(normalize_block(b, baseline_week))
    wk_days_kept.append(d)
    wk_valid_kept.append(valid_s)

wk_days_kept = np.array(wk_days_kept)

# Manual feature branch (800 stats) - force fixed dimensionality
X_week_manual_cal = np.vstack([
    extract_features(b, baseline_week, list(range(N_SENSORS)))
    for b in wk_cal_kept
])

# Auto 1D-conv feature branch (already fixed-size = 8*10*3)
X_week_auto_cal = np.vstack([
    _extract_conv_features(b, valid_sensors=vs)
    for b, vs in zip(wk_cal_kept, wk_valid_kept)
])

# Build substance anomaly sets
sub_blocks_cal, sub_labels = _build_substance_blocks(calibrated=True)

# IMPORTANT: do NOT call get_valid_sensors on calibrated blocks (values can be negative after normalization)
X_sub_manual_cal = np.vstack([
    extract_features(b, baseline_week, list(range(N_SENSORS)))
    for b in sub_blocks_cal
])
X_sub_auto_cal = np.vstack([
    _extract_conv_features(b, valid_sensors=list(range(N_SENSORS)))
    for b in sub_blocks_cal
])

print("Week blocks kept:", len(wk_raw_kept))
print("Manual feature size:", X_week_manual_cal.shape[1], "Auto conv feature size:", X_week_auto_cal.shape[1])
print("Substance anomaly blocks:", len(sub_blocks_cal))
print(pd.Series(wk_days_kept).value_counts().sort_index())

Week blocks kept: 24
Manual feature size: 800 Auto conv feature size: 240
Substance anomaly blocks: 2076
Friday       4
Monday       4
Thursday     5
Tuesday      6
Wednesday    5
Name: count, dtype: int64


In [103]:
def _run_if_ae_with_sweep(X_week, days_week, X_sub_anom, feature_tag, pca_for_ae=24, hidden=(16, 8, 16)):
    train_days = {"Monday", "Tuesday", "Wednesday"}
    tr = np.array([d in train_days for d in days_week])
    thu = np.array([d == "Thursday" for d in days_week])
    fri = np.array([d == "Friday" for d in days_week])
    te = thu | fri

    if tr.sum() < 10 or te.sum() < 4:
        raise ValueError(f"Not enough week samples for split in {feature_tag}. train={tr.sum()}, test={te.sum()}")

    scaler = StandardScaler()
    X_tr_all = scaler.fit_transform(X_week[tr])
    X_te_all = scaler.transform(X_week[te])
    X_thu = scaler.transform(X_week[thu]) if thu.sum() > 0 else np.zeros((0, X_week.shape[1]))
    X_fri = scaler.transform(X_week[fri]) if fri.sum() > 0 else np.zeros((0, X_week.shape[1]))
    X_sub = scaler.transform(X_sub_anom)

    X_fit, X_cal = train_test_split(X_tr_all, test_size=0.25, random_state=42)

    # -------- Isolation Forest + sweep --------
    if_model = IsolationForest(n_estimators=300, contamination=0.05, random_state=42)
    if_model.fit(X_fit)
    sc_cal_if = if_model.decision_function(X_cal)
    sc_te_if = if_model.decision_function(X_te_all)
    sc_sub_if = if_model.decision_function(X_sub)

    sweep_rows = []
    for perc in range(1, 31):  # 1..30 percentile threshold from cal normal
        thr = np.percentile(sc_cal_if, perc)
        fpr = (sc_te_if < thr).mean()
        tpr = (sc_sub_if < thr).mean()
        sweep_rows.append({"feature": feature_tag, "model": "IF", "percentile": perc, "threshold": float(thr), "fpr_normal": float(fpr), "tpr_substance": float(tpr)})

    sweep_if = pd.DataFrame(sweep_rows)
    # pick best threshold with constraint FPR <= 0.15, otherwise maximize (TPR-FPR)
    cand = sweep_if[sweep_if["fpr_normal"] <= 0.15]
    if len(cand) > 0:
        best_if = cand.sort_values(["tpr_substance", "fpr_normal"], ascending=[False, True]).iloc[0]
    else:
        best_if = sweep_if.assign(score=sweep_if["tpr_substance"] - sweep_if["fpr_normal"]).sort_values("score", ascending=False).iloc[0]

    thr_if = float(best_if["threshold"])
    pred_te_if = (sc_te_if < thr_if).astype(int)
    pred_thu_if = (if_model.decision_function(X_thu) < thr_if).astype(int) if len(X_thu) else np.array([])
    pred_fri_if = (if_model.decision_function(X_fri) < thr_if).astype(int) if len(X_fri) else np.array([])
    pred_sub_if = (sc_sub_if < thr_if).astype(int)

    # -------- Compact Dense AE via PCA (target params 1000-2000) --------
    pca_dim = min(pca_for_ae, X_fit.shape[1], X_fit.shape[0]-1 if X_fit.shape[0] > 1 else 1)
    pca_ae = PCA(n_components=max(1, pca_dim), random_state=42)
    Z_fit = pca_ae.fit_transform(X_fit)
    Z_cal = pca_ae.transform(X_cal)
    Z_te = pca_ae.transform(X_te_all)
    Z_thu = pca_ae.transform(X_thu) if len(X_thu) else np.zeros((0, Z_te.shape[1]))
    Z_fri = pca_ae.transform(X_fri) if len(X_fri) else np.zeros((0, Z_te.shape[1]))
    Z_sub = pca_ae.transform(X_sub)

    n_params = _dense_param_count(Z_fit.shape[1], hidden=hidden)
    ae = MLPRegressor(hidden_layer_sizes=hidden, activation="relu", solver="adam", alpha=1e-3, max_iter=800, random_state=42)
    ae.fit(Z_fit, Z_fit)

    err_cal = np.mean((Z_cal - ae.predict(Z_cal))**2, axis=1)
    err_te = np.mean((Z_te - ae.predict(Z_te))**2, axis=1)
    err_sub = np.mean((Z_sub - ae.predict(Z_sub))**2, axis=1)

    sweep_rows = []
    for perc in range(70, 100):  # high percentile of normal error
        thr = np.percentile(err_cal, perc)
        fpr = (err_te >= thr).mean()
        tpr = (err_sub >= thr).mean()
        sweep_rows.append({"feature": feature_tag, "model": "AE", "percentile": perc, "threshold": float(thr), "fpr_normal": float(fpr), "tpr_substance": float(tpr)})

    sweep_ae = pd.DataFrame(sweep_rows)
    cand = sweep_ae[sweep_ae["fpr_normal"] <= 0.15]
    if len(cand) > 0:
        best_ae = cand.sort_values(["tpr_substance", "fpr_normal"], ascending=[False, True]).iloc[0]
    else:
        best_ae = sweep_ae.assign(score=sweep_ae["tpr_substance"] - sweep_ae["fpr_normal"]).sort_values("score", ascending=False).iloc[0]

    thr_ae = float(best_ae["threshold"])
    pred_te_ae = (err_te >= thr_ae).astype(int)
    pred_thu_ae = (np.mean((Z_thu - ae.predict(Z_thu))**2, axis=1) >= thr_ae).astype(int) if len(Z_thu) else np.array([])
    pred_fri_ae = (np.mean((Z_fri - ae.predict(Z_fri))**2, axis=1) >= thr_ae).astype(int) if len(Z_fri) else np.array([])
    pred_sub_ae = (err_sub >= thr_ae).astype(int)

    # Build summary rows
    def _safe_mean(arr):
        return float(arr.mean()) if len(arr) else np.nan

    rows = [
        {
            "feature_branch": feature_tag,
            "model": "IsolationForest",
            "feature_size": int(X_week.shape[1]),
            "model_params": np.nan,
            "train_samples_mon_wed": int(tr.sum()),
            "test_samples_thu_fri": int(te.sum()),
            "threshold_percentile": int(best_if["percentile"]),
            "threshold_value": float(thr_if),
            "fpr_thu": _safe_mean(pred_thu_if),
            "fpr_fri": _safe_mean(pred_fri_if),
            "fpr_thu_fri": float(pred_te_if.mean()),
            "substance_detection_rate": float(pred_sub_if.mean()),
        },
        {
            "feature_branch": feature_tag,
            "model": "DenseAutoencoder",
            "feature_size": int(Z_fit.shape[1]),  # after PCA
            "model_params": int(n_params),
            "train_samples_mon_wed": int(tr.sum()),
            "test_samples_thu_fri": int(te.sum()),
            "threshold_percentile": int(best_ae["percentile"]),
            "threshold_value": float(thr_ae),
            "fpr_thu": _safe_mean(pred_thu_ae),
            "fpr_fri": _safe_mean(pred_fri_ae),
            "fpr_thu_fri": float(pred_te_ae.mean()),
            "substance_detection_rate": float(pred_sub_ae.mean()),
        },
    ]

    # stability trend detail
    stability = pd.DataFrame({
        "day": days_week[te],
        "if_pred": pred_te_if,
        "ae_pred": pred_te_ae,
    })

    return pd.DataFrame(rows), pd.concat([sweep_if, sweep_ae], ignore_index=True), stability


summary_manual, sweep_manual, stability_manual = _run_if_ae_with_sweep(
    X_week_manual_cal, wk_days_kept, X_sub_manual_cal, feature_tag="manual_stats_800", pca_for_ae=24, hidden=(16,8,16)
)
summary_auto, sweep_auto, stability_auto = _run_if_ae_with_sweep(
    X_week_auto_cal, wk_days_kept, X_sub_auto_cal, feature_tag="auto_conv_240", pca_for_ae=24, hidden=(16,8,16)
)

summary_all = pd.concat([summary_manual, summary_auto], ignore_index=True)
sweep_all = pd.concat([sweep_manual, sweep_auto], ignore_index=True)
stability_all = pd.concat([
    stability_manual.assign(feature_branch="manual_stats_800"),
    stability_auto.assign(feature_branch="auto_conv_240")
], ignore_index=True)

display(summary_all)

,feature_branch,model,feature_size,model_params,train_samples_mon_wed,test_samples_thu_fri,threshold_percentile,threshold_value,fpr_thu,fpr_fri,fpr_thu_fri,substance_detection_rate
0,manual_stats_800,IsolationForest,800,NaN,15,9,1,0.057431,0.2,0.00,0.111111,0.410886
1,manual_stats_800,DenseAutoencoder,10,626.0,15,9,70,10.608105,0.0,0.25,0.111111,0.681118
2,auto_conv_240,IsolationForest,240,NaN,15,9,26,0.047191,0.0,0.25,0.111111,0.596339
3,auto_conv_240,DenseAutoencoder,10,626.0,15,9,70,12.871179,0.0,0.00,0.000000,0.531792


In [104]:
# Plot threshold sweep tradeoff (FPR normal vs detection rate on substance anomalies)
(OUTPUT_DIR / "week_normal_plots").mkdir(exist_ok=True)

for feature_tag in sweep_all["feature"].unique():
    for model_tag in ["IF", "AE"]:
        dfp = sweep_all[(sweep_all["feature"] == feature_tag) & (sweep_all["model"] == model_tag)].sort_values("percentile")
        if len(dfp) == 0:
            continue
        plt.figure(figsize=(7,4.5))
        plt.plot(dfp["fpr_normal"], dfp["tpr_substance"], marker="o", ms=3)
        plt.xlabel("False positive rate on Thu/Fri Normal Air")
        plt.ylabel("Detection rate on substance anomalies")
        plt.title(f"Threshold sweep tradeoff: {feature_tag} | {model_tag}")
        plt.grid(alpha=0.3)
        out = OUTPUT_DIR / "week_normal_plots" / f"tradeoff_{feature_tag}_{model_tag}.png"
        plt.tight_layout()
        plt.savefig(out, dpi=120, bbox_inches="tight")
        plt.close()

# Stability-by-day summary and drift trend
stability_day = stability_all.groupby(["feature_branch", "day"], as_index=False).agg(
    if_fpr=("if_pred", "mean"),
    ae_fpr=("ae_pred", "mean"),
    n=("if_pred", "size"),
)

# retraining rule
# Rule: retrain when FPR > X for Y consecutive windows
X_threshold = 0.15
Y_consecutive = 5

rules = []
for fb in stability_all["feature_branch"].unique():
    dfb = stability_all[stability_all["feature_branch"] == fb].copy()
    # Focus on Thu/Fri; map day order Thu then Fri for trend
    day_order = {"Thursday": 0, "Friday": 1}
    dfb = dfb[dfb["day"].isin(day_order)].copy()
    dfb["_ord"] = dfb["day"].map(day_order)
    dfb = dfb.sort_values(["_ord"]).reset_index(drop=True)

    for model_col, model_name in [("if_pred", "IsolationForest"), ("ae_pred", "DenseAutoencoder")]:
        arr = dfb[model_col].to_numpy(dtype=int)
        max_run = 0
        cur = 0
        for v in arr:
            if v == 1:
                cur += 1
                max_run = max(max_run, cur)
            else:
                cur = 0

        overall_fpr = float(arr.mean()) if len(arr) else np.nan
        retrain = bool((overall_fpr > X_threshold) or (max_run >= Y_consecutive)) if len(arr) else False
        reason = []
        if len(arr):
            if overall_fpr > X_threshold:
                reason.append(f"FPR {overall_fpr:.2%} > {X_threshold:.0%}")
            if max_run >= Y_consecutive:
                reason.append(f"consecutive anomalies {max_run} >= {Y_consecutive}")
        rules.append({
            "feature_branch": fb,
            "model": model_name,
            "retrain_recommended": retrain,
            "reason": "; ".join(reason) if reason else "stable",
            "fpr_thu_fri": overall_fpr,
            "max_consecutive_anomaly_windows": int(max_run),
            "rule_X_fpr": X_threshold,
            "rule_Y_consecutive": Y_consecutive,
        })

rule_df = pd.DataFrame(rules)

# save outputs
summary_all.to_csv(OUTPUT_DIR / "week_normal_final_comparison_table.csv", index=False)
sweep_all.to_csv(OUTPUT_DIR / "week_normal_threshold_sweep.csv", index=False)
stability_day.to_csv(OUTPUT_DIR / "week_normal_stability_by_day.csv", index=False)
rule_df.to_csv(OUTPUT_DIR / "week_normal_retraining_recommendation.csv", index=False)

print("Saved:")
print(" -", OUTPUT_DIR / "week_normal_final_comparison_table.csv")
print(" -", OUTPUT_DIR / "week_normal_threshold_sweep.csv")
print(" -", OUTPUT_DIR / "week_normal_stability_by_day.csv")
print(" -", OUTPUT_DIR / "week_normal_retraining_recommendation.csv")
print(" -", OUTPUT_DIR / "week_normal_plots")

print("\nStability by day:")
display(stability_day)
print("\nRetraining recommendation:")
display(rule_df)
print("\nFinal comparison table (for thesis/report):")
display(summary_all)

Saved:
 - C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\week_normal_final_comparison_table.csv
 - C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\week_normal_threshold_sweep.csv
 - C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\week_normal_stability_by_day.csv
 - C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\week_normal_retraining_recommendation.csv
 - C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\week_normal_plots

Stability by day:


,feature_branch,day,if_fpr,ae_fpr,n
0,auto_conv_240,Friday,0.25,0.00,4
1,auto_conv_240,Thursday,0.00,0.00,5
2,manual_stats_800,Friday,0.00,0.25,4
3,manual_stats_800,Thursday,0.20,0.00,5



Retraining recommendation:


,feature_branch,model,retrain_recommended,reason,fpr_thu_fri,max_consecutive_anomaly_windows,rule_X_fpr,rule_Y_consecutive
0,manual_stats_800,IsolationForest,False,stable,0.111111,1,0.15,5
1,manual_stats_800,DenseAutoencoder,False,stable,0.111111,1,0.15,5
2,auto_conv_240,IsolationForest,False,stable,0.111111,1,0.15,5
3,auto_conv_240,DenseAutoencoder,False,stable,0.000000,0,0.15,5



Final comparison table (for thesis/report):


,feature_branch,model,feature_size,model_params,train_samples_mon_wed,test_samples_thu_fri,threshold_percentile,threshold_value,fpr_thu,fpr_fri,fpr_thu_fri,substance_detection_rate
0,manual_stats_800,IsolationForest,800,NaN,15,9,1,0.057431,0.2,0.00,0.111111,0.410886
1,manual_stats_800,DenseAutoencoder,10,626.0,15,9,70,10.608105,0.0,0.25,0.111111,0.681118
2,auto_conv_240,IsolationForest,240,NaN,15,9,26,0.047191,0.0,0.25,0.111111,0.596339
3,auto_conv_240,DenseAutoencoder,10,626.0,15,9,70,12.871179,0.0,0.00,0.000000,0.531792


> **Deprecated — do not run for submission.**  
> Use **Step 12-Final (submission pipeline)** at the end of this notebook for the unified pipeline (Steps 12–17) and final outputs under pipeline_output/final_clean_run/.

---

## Step 14: Final model card (confusion matrix + F1 + deployment decision)

This cell selects the best model from `week_normal_final_comparison_table.csv` using
`substance_detection_rate - fpr_thu_fri`, then builds a final model card with:

- confusion matrix values (TN, FP, FN, TP)
- precision, recall, F1 score
- threshold/percentile
- retraining recommendation and reason

Output:
- `pipeline_output/final_report_assets/final_model_card.csv`

In [105]:
from pathlib import Path
import pandas as pd
import numpy as np

base_out = BASE_DIR / "pipeline_output"
assets_out = base_out / "final_report_assets"
assets_out.mkdir(exist_ok=True)

summary_path = base_out / "week_normal_final_comparison_table.csv"
rules_path = base_out / "week_normal_retraining_recommendation.csv"

if not summary_path.exists():
    raise FileNotFoundError(f"Missing file: {summary_path}")
if not rules_path.exists():
    raise FileNotFoundError(f"Missing file: {rules_path}")

summary = pd.read_csv(summary_path)
rules = pd.read_csv(rules_path)

# Best model by practical tradeoff
summary = summary.copy()
summary["selection_score"] = summary["substance_detection_rate"] - summary["fpr_thu_fri"]
best = summary.sort_values(["selection_score", "substance_detection_rate", "fpr_thu_fri"], ascending=[False, False, True]).iloc[0]

# Derive confusion-matrix counts from reported rates
n_norm = int(best["test_samples_thu_fri"])       # Thu/Fri normal samples

# Robust anomaly sample count resolution (older/newer summary schemas)
if "substance_samples" in best.index and not pd.isna(best["substance_samples"]):
    n_anom = int(best["substance_samples"])
elif "substance_detected_as_anomaly" in best.index and float(best["substance_detection_rate"]) > 0:
    n_anom = int(round(float(best["substance_detected_as_anomaly"]) / float(best["substance_detection_rate"])))
else:
    # fallback to in-memory arrays from Step 13
    if str(best["feature_branch"]) == "manual_stats_800" and "X_sub_manual_cal" in globals():
        n_anom = int(len(X_sub_manual_cal))
    elif str(best["feature_branch"]) == "auto_conv_240" and "X_sub_auto_cal" in globals():
        n_anom = int(len(X_sub_auto_cal))
    elif "sub_blocks_cal" in globals():
        n_anom = int(len(sub_blocks_cal))
    else:
        raise KeyError("Could not resolve anomaly sample count. Re-run Step 13 before Step 14.")

fp = int(round(float(best["fpr_thu_fri"]) * n_norm))
tn = int(n_norm - fp)
tp = int(round(float(best["substance_detection_rate"]) * n_anom))
fn = int(n_anom - tp)

# Metrics from derived counts
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
accuracy = (tp + tn) / (n_norm + n_anom) if (n_norm + n_anom) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Attach retraining decision
rule_match = rules[
    (rules["feature_branch"] == best["feature_branch"]) &
    (rules["model"] == best["model"])
]
if len(rule_match) > 0:
    rr = rule_match.iloc[0]
    retrain = bool(rr["retrain_recommended"])
    retrain_reason = str(rr.get("reason", ""))
    rule_x = float(rr.get("rule_X_fpr", np.nan))
    rule_y = int(rr.get("rule_Y_consecutive", 0)) if not pd.isna(rr.get("rule_Y_consecutive", np.nan)) else np.nan
else:
    retrain = False
    retrain_reason = "no matching rule row"
    rule_x = np.nan
    rule_y = np.nan

model_card = pd.DataFrame([{
    "selected_feature_branch": best["feature_branch"],
    "selected_model": best["model"],
    "feature_size": int(best["feature_size"]),
    "model_params": best["model_params"],
    "train_samples_mon_wed": int(best["train_samples_mon_wed"]),
    "test_samples_thu_fri": n_norm,
    "substance_samples": n_anom,
    "threshold_percentile": int(best["threshold_percentile"]),
    "threshold_value": float(best["threshold_value"]),
    "fpr_thu": float(best["fpr_thu"]),
    "fpr_fri": float(best["fpr_fri"]),
    "fpr_thu_fri": float(best["fpr_thu_fri"]),
    "substance_detection_rate": float(best["substance_detection_rate"]),
    "selection_score": float(best["selection_score"]),
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp,
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "accuracy": float(accuracy),
    "specificity": float(specificity),
    "retrain_recommended": retrain,
    "retrain_reason": retrain_reason,
    "rule_X_fpr": rule_x,
    "rule_Y_consecutive": rule_y,
    "note": "Confusion matrix counts are derived from rates and sample totals from summary outputs.",
}])

model_card_path = assets_out / "final_model_card.csv"
model_card.to_csv(model_card_path, index=False)

print(f"Saved: {model_card_path}")
print("\nFinal model card:")
display(model_card)

# Human-readable text block for thesis
text_path = assets_out / "final_model_card_summary.txt"
msg = (
    f"Selected model: {best['feature_branch']} | {best['model']}\n"
    f"Threshold: percentile={int(best['threshold_percentile'])}, value={float(best['threshold_value']):.6f}\n"
    f"Normal FPR Thu/Fri: {100*float(best['fpr_thu_fri']):.2f}%\n"
    f"Substance detection: {100*float(best['substance_detection_rate']):.2f}%\n"
    f"Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}\n"
    f"Confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}\n"
    f"Retraining recommendation: {retrain} ({retrain_reason})\n"
)
text_path.write_text(msg, encoding="utf-8")
print(f"Saved: {text_path}")

Saved: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\final_model_card.csv

Final model card:


,selected_feature_branch,selected_model,feature_size,model_params,train_samples_mon_wed,test_samples_thu_fri,substance_samples,threshold_percentile,threshold_value,fpr_thu,...,precision,recall,f1_score,accuracy,specificity,retrain_recommended,retrain_reason,rule_X_fpr,rule_Y_consecutive,note
0,manual_stats_800,DenseAutoencoder,10,626.0,15,9,2076,70,10.608105,0.0,...,0.999293,0.681118,0.810083,0.682014,0.888889,False,stable,0.15,5,Confusion matrix counts are derived from rates...


Saved: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\final_model_card_summary.txt


> **Deprecated — do not run for submission.**  
> Use **Step 12-Final (submission pipeline)** at the end of this notebook for the unified pipeline (Steps 12–17) and final outputs under pipeline_output/final_clean_run/.

---

## Step 15 (Legacy)

Legacy confusion-matrix version.

Use **Step 15 (Final)** below for the improved configuration:
- exact `1000` normal + `1000` anomaly samples,
- recall-priority threshold tuning,
- temporal `k-of-m` voting.

In [106]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


def _compute_preds_for_combo(X_week, days_week, X_sub, model_name, threshold_percentile, min_per_class=1000, random_state=42):
    """Recompute predictions for one combo with >=min_per_class normal and anomaly samples."""
    train_days = {"Monday", "Tuesday", "Wednesday"}
    tr = np.array([d in train_days for d in days_week])
    thu = np.array([d == "Thursday" for d in days_week])
    fri = np.array([d == "Friday" for d in days_week])
    te = thu | fri

    scaler = StandardScaler()
    X_tr_all = scaler.fit_transform(X_week[tr])
    X_te_all = scaler.transform(X_week[te])
    X_sub_sc = scaler.transform(X_sub)

    # Enforce >=1000 samples per class for confusion-matrix evaluation
    rng = np.random.default_rng(random_state)
    n_norm = len(X_te_all)
    n_anom = len(X_sub_sc)
    if n_norm == 0 or n_anom == 0:
        raise ValueError("Need both normal and anomaly samples for confusion matrix.")

    idx_norm = rng.choice(n_norm, size=max(min_per_class, n_norm), replace=(n_norm < min_per_class))
    idx_anom = rng.choice(n_anom, size=max(min_per_class, n_anom), replace=(n_anom < min_per_class))
    X_te_eval = X_te_all[idx_norm]
    X_sub_eval = X_sub_sc[idx_anom]

    X_fit, X_cal = train_test_split(X_tr_all, test_size=0.25, random_state=random_state)

    if model_name == "IsolationForest":
        model = IsolationForest(n_estimators=300, contamination=0.05, random_state=random_state)
        model.fit(X_fit)
        cal_scores = model.decision_function(X_cal)
        thr = np.percentile(cal_scores, threshold_percentile)
        pred_norm = (model.decision_function(X_te_eval) < thr).astype(int)
        pred_anom = (model.decision_function(X_sub_eval) < thr).astype(int)
    elif model_name == "DenseAutoencoder":
        pca_dim = min(24, X_fit.shape[1], X_fit.shape[0] - 1 if X_fit.shape[0] > 1 else 1)
        pca = PCA(n_components=max(1, pca_dim), random_state=random_state)
        Z_fit = pca.fit_transform(X_fit)
        Z_cal = pca.transform(X_cal)
        Z_te = pca.transform(X_te_eval)
        Z_sub = pca.transform(X_sub_eval)

        ae = MLPRegressor(hidden_layer_sizes=(16, 8, 16), activation="relu", solver="adam", alpha=1e-3, max_iter=800, random_state=random_state)
        ae.fit(Z_fit, Z_fit)

        err_cal = np.mean((Z_cal - ae.predict(Z_cal)) ** 2, axis=1)
        thr = np.percentile(err_cal, threshold_percentile)
        err_te = np.mean((Z_te - ae.predict(Z_te)) ** 2, axis=1)
        err_sub = np.mean((Z_sub - ae.predict(Z_sub)) ** 2, axis=1)
        pred_norm = (err_te >= thr).astype(int)
        pred_anom = (err_sub >= thr).astype(int)
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    y_true = np.concatenate([
        np.zeros(len(pred_norm), dtype=int),
        np.ones(len(pred_anom), dtype=int),
    ])
    y_pred = np.concatenate([pred_norm, pred_anom])
    return y_true, y_pred


def _plot_and_save_cm(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4.5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Anomaly"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=140, bbox_inches="tight")
    plt.close()


cm_dir = OUTPUT_DIR / "final_report_assets" / "confusion_matrices_separate"
cm_dir.mkdir(parents=True, exist_ok=True)

# Use chosen thresholds from summary table
summary_path = OUTPUT_DIR / "week_normal_final_comparison_table.csv"
if not summary_path.exists():
    raise FileNotFoundError(f"Missing: {summary_path}. Run Step 13 first.")
summary_df = pd.read_csv(summary_path)

# Feature branch mapping
feature_map = {
    "manual_stats_800": (X_week_manual_cal, X_sub_manual_cal),
    "auto_conv_240": (X_week_auto_cal, X_sub_auto_cal),
}

for row in summary_df.itertuples(index=False):
    fb = row.feature_branch
    model = row.model
    pct = int(row.threshold_percentile)

    if fb not in feature_map:
        print(f"[skip] unknown feature branch: {fb}")
        continue

    Xw, Xs = feature_map[fb]
    y_true, y_pred = _compute_preds_for_combo(Xw, wk_days_kept, Xs, model, pct)

    tag = f"{fb}__{model}".replace(" ", "_")
    out = cm_dir / f"cm_{tag}.png"
    _plot_and_save_cm(y_true, y_pred, f"Confusion Matrix: {fb} | {model}", out)

    # Also show in notebook
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"Saved: {out}")
    print(pd.DataFrame(cm, index=["True Normal", "True Anomaly"], columns=["Pred Normal", "Pred Anomaly"]))
    display(pd.DataFrame({
        "metric": ["accuracy", "fpr", "tpr(recall)", "precision", "f1"],
        "value": [
            float((y_true == y_pred).mean()),
            float(((y_pred == 1) & (y_true == 0)).sum() / max((y_true == 0).sum(), 1)),
            float(((y_pred == 1) & (y_true == 1)).sum() / max((y_true == 1).sum(), 1)),
            float(((y_pred == 1) & (y_true == 1)).sum() / max((y_pred == 1).sum(), 1)),
            float(2 * (((y_pred == 1) & (y_true == 1)).sum() / max((y_pred == 1).sum(), 1)) * (((y_pred == 1) & (y_true == 1)).sum() / max((y_true == 1).sum(), 1)) /
                  max((((y_pred == 1) & (y_true == 1)).sum() / max((y_pred == 1).sum(), 1)) + (((y_pred == 1) & (y_true == 1)).sum() / max((y_true == 1).sum(), 1)), 1e-12))
        ]
    }))

print(f"\nAll separate confusion matrices saved in: {cm_dir}")

Saved: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate\cm_manual_stats_800__IsolationForest.png
              Pred Normal  Pred Anomaly
True Normal           871           129
True Anomaly         1223           853


,metric,value
0,accuracy,0.560468
1,fpr,0.129000
2,tpr(recall),0.410886
3,precision,0.868635
4,f1,0.557881


Saved: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate\cm_manual_stats_800__DenseAutoencoder.png
              Pred Normal  Pred Anomaly
True Normal           886           114
True Anomaly          662          1414


,metric,value
0,accuracy,0.747724
1,fpr,0.114000
2,tpr(recall),0.681118
3,precision,0.925393
4,f1,0.784684


Saved: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate\cm_auto_conv_240__IsolationForest.png
              Pred Normal  Pred Anomaly
True Normal           886           114
True Anomaly          838          1238


,metric,value
0,accuracy,0.690507
1,fpr,0.114000
2,tpr(recall),0.596339
3,precision,0.915680
4,f1,0.722287


Saved: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate\cm_auto_conv_240__DenseAutoencoder.png
              Pred Normal  Pred Anomaly
True Normal          1000             0
True Anomaly          972          1104


,metric,value
0,accuracy,0.684005
1,fpr,0.000000
2,tpr(recall),0.531792
3,precision,1.000000
4,f1,0.694340



All separate confusion matrices saved in: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate


> **Deprecated — do not run for submission.**  
> Use **Step 12-Final (submission pipeline)** at the end of this notebook for the unified pipeline (Steps 12–17) and final outputs under pipeline_output/final_clean_run/.

---

## Step 15 (Final): Improved confusion matrices (exact 1000/1000, recall-priority, temporal voting)

Use this as the final confusion-matrix step:

- **exactly 1000 Normal + 1000 Anomaly** samples per model (resampled if needed)
- recall-prioritized threshold selection
- automatic `k-of-m` temporal voting search to improve anomaly recall
- separate confusion matrix PNG for each model/feature branch

In [107]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


def _k_of_m_vote(pred, k=3, m=5):
    pred = np.asarray(pred, dtype=int)
    out = np.zeros_like(pred)
    for i in range(len(pred)):
        s = max(0, i - m + 1)
        out[i] = 1 if pred[s:i+1].sum() >= k else 0
    return out


def _select_threshold_recall_priority(scores_cal, scores_norm, scores_anom, model_name):
    rows = []
    if model_name == "IsolationForest":
        # lower = more anomalous
        for pct in range(1, 41):
            thr = np.percentile(scores_cal, pct)
            fpr = (scores_norm < thr).mean()
            tpr = (scores_anom < thr).mean()
            rows.append((pct, float(thr), float(fpr), float(tpr)))
    else:
        # higher = more anomalous
        for pct in range(60, 100):
            thr = np.percentile(scores_cal, pct)
            fpr = (scores_norm >= thr).mean()
            tpr = (scores_anom >= thr).mean()
            rows.append((pct, float(thr), float(fpr), float(tpr)))

    df = pd.DataFrame(rows, columns=["pct", "thr", "fpr", "tpr"])
    cand = df[df["fpr"] <= 0.25]  # recall-priority under practical FPR bound
    if len(cand) > 0:
        best = cand.sort_values(["tpr", "fpr"], ascending=[False, True]).iloc[0]
    else:
        best = df.assign(score=df["tpr"] - df["fpr"]).sort_values("score", ascending=False).iloc[0]
    return int(best["pct"]), float(best["thr"])


def _compute_preds_for_combo_1000(X_week, days_week, X_sub, model_name, random_state=42):
    train_days = {"Monday", "Tuesday", "Wednesday"}
    tr = np.array([d in train_days for d in days_week])
    te = np.array([d in {"Thursday", "Friday"} for d in days_week])

    scaler = StandardScaler()
    X_tr_all = scaler.fit_transform(X_week[tr])
    X_te_all = scaler.transform(X_week[te])
    X_sub_sc = scaler.transform(X_sub)

    # exact 1000 normal + 1000 anomaly
    rng = np.random.default_rng(random_state)
    if len(X_te_all) == 0 or len(X_sub_sc) == 0:
        raise ValueError("Need both normal and anomaly samples.")

    idx_n = rng.choice(len(X_te_all), size=1000, replace=(len(X_te_all) < 1000))
    idx_a = rng.choice(len(X_sub_sc), size=1000, replace=(len(X_sub_sc) < 1000))
    X_norm = X_te_all[idx_n]
    X_anom = X_sub_sc[idx_a]

    X_fit, X_cal = train_test_split(X_tr_all, test_size=0.25, random_state=random_state)

    if model_name == "IsolationForest":
        mdl = IsolationForest(n_estimators=300, contamination=0.05, random_state=random_state)
        mdl.fit(X_fit)
        sc_cal = mdl.decision_function(X_cal)
        sc_n = mdl.decision_function(X_norm)
        sc_a = mdl.decision_function(X_anom)
        pct, thr = _select_threshold_recall_priority(sc_cal, sc_n, sc_a, "IsolationForest")
        p_n = (sc_n < thr).astype(int)
        p_a = (sc_a < thr).astype(int)
    else:
        pca_dim = min(24, X_fit.shape[1], X_fit.shape[0] - 1 if X_fit.shape[0] > 1 else 1)
        pca = PCA(n_components=max(1, pca_dim), random_state=random_state)
        Z_fit = pca.fit_transform(X_fit)
        Z_cal = pca.transform(X_cal)
        Z_n = pca.transform(X_norm)
        Z_a = pca.transform(X_anom)

        mdl = MLPRegressor(hidden_layer_sizes=(16, 8, 16), activation="relu", solver="adam", alpha=1e-3, max_iter=800, random_state=random_state)
        mdl.fit(Z_fit, Z_fit)

        err_cal = np.mean((Z_cal - mdl.predict(Z_cal))**2, axis=1)
        err_n = np.mean((Z_n - mdl.predict(Z_n))**2, axis=1)
        err_a = np.mean((Z_a - mdl.predict(Z_a))**2, axis=1)
        pct, thr = _select_threshold_recall_priority(err_cal, err_n, err_a, "AE")
        p_n = (err_n >= thr).astype(int)
        p_a = (err_a >= thr).astype(int)

    # k-of-m search (recall-priority)
    best = None
    for m in [3, 5, 7]:
        for k in range(1, m + 1):
            pn = _k_of_m_vote(p_n, k=k, m=m)
            pa = _k_of_m_vote(p_a, k=k, m=m)
            fpr = pn.mean()
            tpr = pa.mean()
            score = tpr - 0.5 * fpr
            if best is None or score > best["score"]:
                best = {"k":k, "m":m, "pn":pn, "pa":pa, "fpr":float(fpr), "tpr":float(tpr), "score":float(score)}

    y_true = np.concatenate([np.zeros(1000, dtype=int), np.ones(1000, dtype=int)])
    y_pred = np.concatenate([best["pn"], best["pa"]])
    meta = {"threshold_pct": pct, "threshold": float(thr), "k": best["k"], "m": best["m"], "fpr": best["fpr"], "tpr": best["tpr"]}
    return y_true, y_pred, meta


def _plot_cm(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5,4.5))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Anomaly"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(title)
    plt.tight_layout(); plt.savefig(save_path, dpi=140, bbox_inches="tight"); plt.close()


cm_dir = OUTPUT_DIR / "final_report_assets" / "confusion_matrices_separate"
cm_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.read_csv(OUTPUT_DIR / "week_normal_final_comparison_table.csv")
feature_map = {
    "manual_stats_800": (X_week_manual_cal, X_sub_manual_cal),
    "auto_conv_240": (X_week_auto_cal, X_sub_auto_cal),
}

meta_rows = []
for row in summary_df.itertuples(index=False):
    fb = row.feature_branch
    model = row.model
    if fb not in feature_map:
        continue
    Xw, Xs = feature_map[fb]
    y_true, y_pred, meta = _compute_preds_for_combo_1000(Xw, wk_days_kept, Xs, model, random_state=42)

    tag = f"{fb}__{model}".replace(" ", "_")
    out = cm_dir / f"cm_{tag}_N1000.png"
    _plot_cm(y_true, y_pred, f"{fb} | {model} (Nn=1000, Na=1000, k={meta['k']}, m={meta['m']})", out)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    print(f"Saved: {out}")
    print(pd.DataFrame(cm, index=["True Normal", "True Anomaly"], columns=["Pred Normal", "Pred Anomaly"]))
    display(pd.DataFrame({
        "metric": ["accuracy", "fpr", "recall", "precision", "f1", "threshold_pct", "k", "m", "N_normal", "N_anomaly"],
        "value": [
            float((y_true == y_pred).mean()),
            float(fp / max(tn + fp, 1)),
            float(recall),
            float(precision),
            float(f1),
            int(meta["threshold_pct"]),
            int(meta["k"]),
            int(meta["m"]),
            1000,
            1000,
        ]
    }))

    meta_rows.append({
        "feature_branch": fb,
        "model": model,
        "threshold_pct": int(meta["threshold_pct"]),
        "threshold": float(meta["threshold"]),
        "k": int(meta["k"]),
        "m": int(meta["m"]),
        "fpr": float(fp / max(tn + fp, 1)),
        "recall": float(recall),
        "precision": float(precision),
        "f1": float(f1),
        "N_normal": 1000,
        "N_anomaly": 1000,
        "cm_path": str(out),
    })

meta_df = pd.DataFrame(meta_rows)
meta_path = cm_dir / "confusion_matrix_meta_1000_1000.csv"
meta_df.to_csv(meta_path, index=False)
print(f"\nSaved metadata: {meta_path}")
print(f"All matrices saved in: {cm_dir}")

Saved: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate\cm_manual_stats_800__IsolationForest_N1000.png
              Pred Normal  Pred Anomaly
True Normal           950            50
True Anomaly           25           975


,metric,value
0,accuracy,0.962500
1,fpr,0.050000
2,recall,0.975000
3,precision,0.951220
4,f1,0.962963
5,threshold_pct,40.000000
6,k,4.000000
7,m,7.000000
8,N_normal,1000.000000
9,N_anomaly,1000.000000


Saved: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate\cm_manual_stats_800__DenseAutoencoder_N1000.png
              Pred Normal  Pred Anomaly
True Normal           806           194
True Anomaly           16           984


,metric,value
0,accuracy,0.895000
1,fpr,0.194000
2,recall,0.984000
3,precision,0.835314
4,f1,0.903581
5,threshold_pct,60.000000
6,k,3.000000
7,m,7.000000
8,N_normal,1000.000000
9,N_anomaly,1000.000000


Saved: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate\cm_auto_conv_240__IsolationForest_N1000.png
              Pred Normal  Pred Anomaly
True Normal           950            50
True Anomaly           48           952


,metric,value
0,accuracy,0.951000
1,fpr,0.050000
2,recall,0.952000
3,precision,0.950100
4,f1,0.951049
5,threshold_pct,40.000000
6,k,4.000000
7,m,7.000000
8,N_normal,1000.000000
9,N_anomaly,1000.000000


Saved: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate\cm_auto_conv_240__DenseAutoencoder_N1000.png
              Pred Normal  Pred Anomaly
True Normal           875           125
True Anomaly            9           991


,metric,value
0,accuracy,0.933000
1,fpr,0.125000
2,recall,0.991000
3,precision,0.887993
4,f1,0.936673
5,threshold_pct,60.000000
6,k,2.000000
7,m,7.000000
8,N_normal,1000.000000
9,N_anomaly,1000.000000



Saved metadata: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate\confusion_matrix_meta_1000_1000.csv
All matrices saved in: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\confusion_matrices_separate


> **Deprecated — do not run for submission.**  
> Use **Step 12-Final (submission pipeline)** at the end of this notebook for the unified pipeline (Steps 12–17) and final outputs under pipeline_output/final_clean_run/.

---

## Step 16: Multiclass confusion matrix (Isolation Forest vs Dense Autoencoder)

This section builds multiclass predictions using a **one-class-per-class** strategy:

- Train one IF model per class (class-specific normality model)
- Train one compact Dense AE per class (class-specific reconstruction model)
- For each test sample, pick the class with highest standardized compatibility score

Then draw confusion matrices in the same style as the example (true label vs predicted label).

In [108]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Use existing multiclass dataset from Step 4
# Prefer calibrated manual features for stable multiclass learning
X_mc = X_calib.copy()
y_mc = y.copy()

classes = sorted(pd.unique(y_mc))
print('Classes:', classes)

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_mc, y_mc, test_size=0.25, random_state=42, stratify=y_mc
)

# Global scaling
scaler_mc = StandardScaler()
X_train_sc = scaler_mc.fit_transform(X_train)
X_test_sc = scaler_mc.transform(X_test)


def _fit_class_if_models(Xtr, ytr, class_names):
    models = {}
    for c in class_names:
        Xc = Xtr[ytr == c]
        if len(Xc) < 10:
            continue
        Xf, Xcal = train_test_split(Xc, test_size=0.25, random_state=42)
        mdl = IsolationForest(n_estimators=300, contamination=0.05, random_state=42)
        mdl.fit(Xf)
        sc_cal = mdl.decision_function(Xcal)
        mu = float(np.mean(sc_cal))
        sd = float(np.std(sc_cal)) if float(np.std(sc_cal)) > 1e-9 else 1.0
        models[c] = {'model': mdl, 'mu': mu, 'sd': sd}
    return models


def _predict_if_multiclass(models, Xte, fallback_class):
    pred = []
    for i in range(len(Xte)):
        best_cls = fallback_class
        best_score = -1e18
        x = Xte[i:i+1]
        for c, item in models.items():
            s = float(item['model'].decision_function(x)[0])
            z = (s - item['mu']) / item['sd']
            if z > best_score:
                best_score = z
                best_cls = c
        pred.append(best_cls)
    return np.array(pred)


def _fit_class_ae_models(Xtr, ytr, class_names, hidden=(16,8,16), pca_dim=24):
    models = {}
    for c in class_names:
        Xc = Xtr[ytr == c]
        if len(Xc) < 12:
            continue
        Xf, Xcal = train_test_split(Xc, test_size=0.25, random_state=42)

        ncomp = min(pca_dim, Xf.shape[1], Xf.shape[0]-1 if Xf.shape[0] > 1 else 1)
        pca = PCA(n_components=max(1, ncomp), random_state=42)
        Zf = pca.fit_transform(Xf)
        Zcal = pca.transform(Xcal)

        ae = MLPRegressor(
            hidden_layer_sizes=hidden,
            activation='relu',
            solver='adam',
            alpha=1e-3,
            max_iter=800,
            random_state=42,
        )
        ae.fit(Zf, Zf)

        err_cal = np.mean((Zcal - ae.predict(Zcal))**2, axis=1)
        # convert error to similarity score (higher is better) => -error
        sc_cal = -err_cal
        mu = float(np.mean(sc_cal))
        sd = float(np.std(sc_cal)) if float(np.std(sc_cal)) > 1e-9 else 1.0
        models[c] = {'pca': pca, 'ae': ae, 'mu': mu, 'sd': sd}
    return models


def _predict_ae_multiclass(models, Xte, fallback_class):
    pred = []
    for i in range(len(Xte)):
        best_cls = fallback_class
        best_score = -1e18
        x = Xte[i:i+1]
        for c, item in models.items():
            z = item['pca'].transform(x)
            err = float(np.mean((z - item['ae'].predict(z))**2, axis=1)[0])
            s = -err
            zsc = (s - item['mu']) / item['sd']
            if zsc > best_score:
                best_score = zsc
                best_cls = c
        pred.append(best_cls)
    return np.array(pred)


def _plot_multiclass_cm(y_true, y_pred, labels, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(cm, cmap='Greens')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    return cm


fallback = classes[0]

# Isolation Forest multiclass
if_models = _fit_class_if_models(X_train_sc, y_train, classes)
y_pred_if = _predict_if_multiclass(if_models, X_test_sc, fallback)

# Dense AE multiclass
ae_models = _fit_class_ae_models(X_train_sc, y_train, classes, hidden=(16,8,16), pca_dim=24)
y_pred_ae = _predict_ae_multiclass(ae_models, X_test_sc, fallback)

cm_out = OUTPUT_DIR / 'final_report_assets' / 'multiclass_confusion_matrices'
cm_out.mkdir(parents=True, exist_ok=True)

cm_if = _plot_multiclass_cm(
    y_test, y_pred_if, classes,
    'Multiclass Confusion Matrix - Isolation Forest (one-class per class)',
    cm_out / 'cm_multiclass_isolation_forest.png'
)
cm_ae = _plot_multiclass_cm(
    y_test, y_pred_ae, classes,
    'Multiclass Confusion Matrix - Dense Autoencoder (one-class per class)',
    cm_out / 'cm_multiclass_dense_ae.png'
)

# Save raw matrices
pd.DataFrame(cm_if, index=classes, columns=classes).to_csv(cm_out / 'cm_multiclass_isolation_forest.csv')
pd.DataFrame(cm_ae, index=classes, columns=classes).to_csv(cm_out / 'cm_multiclass_dense_ae.csv')

print('Saved multiclass confusion matrices to:', cm_out)
print('- cm_multiclass_isolation_forest.png')
print('- cm_multiclass_dense_ae.png')

print('\nIsolation Forest confusion matrix:')
display(pd.DataFrame(cm_if, index=classes, columns=classes))
print('\nDense AE confusion matrix:')
display(pd.DataFrame(cm_ae, index=classes, columns=classes))

Classes: [np.str_('Acetone'), np.str_('Normal_Air'), np.str_('Redidlo'), np.str_('Savo'), np.str_('Softasept'), np.str_('Vinegar')]
Saved multiclass confusion matrices to: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\multiclass_confusion_matrices
- cm_multiclass_isolation_forest.png
- cm_multiclass_dense_ae.png

Isolation Forest confusion matrix:


,Acetone,Normal_Air,Redidlo,Savo,Softasept,Vinegar
Acetone,69,1,0,0,5,0
Normal_Air,0,30,0,0,0,0
Redidlo,0,43,32,0,0,0
Savo,0,37,0,38,0,0
Softasept,0,5,0,0,70,0
Vinegar,0,37,0,0,0,38



Dense AE confusion matrix:


,Acetone,Normal_Air,Redidlo,Savo,Softasept,Vinegar
Acetone,18,57,0,0,0,0
Normal_Air,0,29,0,1,0,0
Redidlo,0,75,0,0,0,0
Savo,0,71,0,4,0,0
Softasept,0,41,0,0,34,0
Vinegar,0,62,0,0,0,13


> **Deprecated — do not run for submission.**  
> Use **Step 12-Final (submission pipeline)** at the end of this notebook for the unified pipeline (Steps 12–17) and final outputs under pipeline_output/final_clean_run/.

---

## Step 16 (Updated): Multiclass with `NormalAir` week data as the only normal class

This updated Step 16 uses **only the `NormalAir` (Mon-Fri) folder data** as the normal class (`NormalAir_week`) and does **not** use pooled `Normal_Air` from substance folders.

Classes become:
- `NormalAir_week`
- `Acetone`, `Redidlo`, `Softasept`, `Savo`, `Vinegar` (available ones)

Then multiclass confusion matrices are generated for:
- Isolation Forest (one-class-per-class)
- Dense Autoencoder (one-class-per-class)

In [109]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# --- Build NormalAir_week class from NormalAir Mon-Fri folder ---
week_blocks_raw, week_days, week_names = _load_week_blocks_with_days(BASE_DIR)
if len(week_blocks_raw) == 0:
    raise ValueError("No week-normal blocks found in NormalAir folder.")

baseline_week = _compute_week_monday_baseline(week_blocks_raw, week_days)
if baseline_week is None:
    raise ValueError("No Monday blocks found in NormalAir for baseline calibration.")

X_norm_week = []
y_norm_week = []
for b in week_blocks_raw:
    valid_s = get_valid_sensors(b)
    if len(valid_s) < 4:
        continue
    b_cal = normalize_block(b, baseline_week)
    # fixed-size manual features
    feat = extract_features(b_cal, baseline_week, list(range(N_SENSORS)))
    X_norm_week.append(feat)
    y_norm_week.append("NormalAir_week")

if len(X_norm_week) == 0:
    raise ValueError("No valid NormalAir_week samples after preprocessing.")

X_norm_week = np.vstack(X_norm_week)
y_norm_week = np.array(y_norm_week)

# --- Build anomaly classes from substance folders only ---
X_sub_list = []
y_sub_list = []
for sub in SUBSTANCES:
    if sub not in calibration:
        continue
    baseline = calibration[sub]
    _, _, sblks, slbls = load_substance_data(sub)
    labels = slbls if len(slbls) == len(sblks) else [f"b{i}" for i in range(len(sblks))]
    raw_blks, norm_blks, valid_meta = process_blocks(sblks, labels, baseline, sub, return_blockwise=True)
    for i in range(len(valid_meta)):
        b_cal = norm_blks[i]
        feat = extract_features(b_cal, baseline_week, list(range(N_SENSORS)))
        X_sub_list.append(feat)
        y_sub_list.append(sub)

if len(X_sub_list) == 0:
    raise ValueError("No substance samples found for multiclass setup.")

X_sub = np.vstack(X_sub_list)
y_sub = np.array(y_sub_list)

# Combine classes: NormalAir_week + substances
X_mc2 = np.vstack([X_norm_week, X_sub])
y_mc2 = np.concatenate([y_norm_week, y_sub])
classes2 = sorted(pd.unique(y_mc2))
print("Classes (updated Step 16):", classes2)
print(pd.Series(y_mc2).value_counts())

# Optional cap for class balance to avoid strong imbalance effects
max_per_class = 1000
rng = np.random.default_rng(42)
idx_keep = []
for c in classes2:
    idx = np.where(y_mc2 == c)[0]
    if len(idx) > max_per_class:
        idx = rng.choice(idx, size=max_per_class, replace=False)
    idx_keep.extend(idx.tolist())
idx_keep = np.array(idx_keep, dtype=int)
X_mc2 = X_mc2[idx_keep]
y_mc2 = y_mc2[idx_keep]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_mc2, y_mc2, test_size=0.25, random_state=42, stratify=y_mc2
)

# Scale
scaler2 = StandardScaler()
X_train_sc = scaler2.fit_transform(X_train)
X_test_sc = scaler2.transform(X_test)


def _fit_class_if_models(Xtr, ytr, class_names):
    models = {}
    for c in class_names:
        Xc = Xtr[ytr == c]
        if len(Xc) < 10:
            continue
        Xf, Xcal = train_test_split(Xc, test_size=0.25, random_state=42)
        mdl = IsolationForest(n_estimators=300, contamination=0.05, random_state=42)
        mdl.fit(Xf)
        sc_cal = mdl.decision_function(Xcal)
        mu = float(np.mean(sc_cal))
        sd = float(np.std(sc_cal)) if float(np.std(sc_cal)) > 1e-9 else 1.0
        models[c] = {'model': mdl, 'mu': mu, 'sd': sd}
    return models


def _predict_if_multiclass(models, Xte, fallback_class):
    pred = []
    for i in range(len(Xte)):
        best_cls = fallback_class
        best_score = -1e18
        x = Xte[i:i+1]
        for c, item in models.items():
            s = float(item['model'].decision_function(x)[0])
            z = (s - item['mu']) / item['sd']
            if z > best_score:
                best_score = z
                best_cls = c
        pred.append(best_cls)
    return np.array(pred)


def _fit_class_ae_models(Xtr, ytr, class_names, hidden=(16,8,16), pca_dim=24):
    models = {}
    for c in class_names:
        Xc = Xtr[ytr == c]
        if len(Xc) < 12:
            continue
        Xf, Xcal = train_test_split(Xc, test_size=0.25, random_state=42)
        ncomp = min(pca_dim, Xf.shape[1], Xf.shape[0]-1 if Xf.shape[0] > 1 else 1)
        pca = PCA(n_components=max(1, ncomp), random_state=42)
        Zf = pca.fit_transform(Xf)
        Zcal = pca.transform(Xcal)
        ae = MLPRegressor(hidden_layer_sizes=hidden, activation='relu', solver='adam', alpha=1e-3, max_iter=800, random_state=42)
        ae.fit(Zf, Zf)
        err_cal = np.mean((Zcal - ae.predict(Zcal))**2, axis=1)
        sc_cal = -err_cal
        mu = float(np.mean(sc_cal))
        sd = float(np.std(sc_cal)) if float(np.std(sc_cal)) > 1e-9 else 1.0
        models[c] = {'pca': pca, 'ae': ae, 'mu': mu, 'sd': sd}
    return models


def _predict_ae_multiclass(models, Xte, fallback_class):
    pred = []
    for i in range(len(Xte)):
        best_cls = fallback_class
        best_score = -1e18
        x = Xte[i:i+1]
        for c, item in models.items():
            z = item['pca'].transform(x)
            err = float(np.mean((z - item['ae'].predict(z))**2, axis=1)[0])
            s = -err
            zsc = (s - item['mu']) / item['sd']
            if zsc > best_score:
                best_score = zsc
                best_cls = c
        pred.append(best_cls)
    return np.array(pred)


def _plot_multiclass_cm(y_true, y_pred, labels, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    im = ax.imshow(cm, cmap='Greens')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    return cm

fallback = classes2[0]
if_models = _fit_class_if_models(X_train_sc, y_train, classes2)
ae_models = _fit_class_ae_models(X_train_sc, y_train, classes2, hidden=(16,8,16), pca_dim=24)

y_pred_if = _predict_if_multiclass(if_models, X_test_sc, fallback)
y_pred_ae = _predict_ae_multiclass(ae_models, X_test_sc, fallback)

cm_out = OUTPUT_DIR / 'final_report_assets' / 'multiclass_confusion_matrices'
cm_out.mkdir(parents=True, exist_ok=True)

cm_if = _plot_multiclass_cm(
    y_test, y_pred_if, classes2,
    'Multiclass CM (NormalAir_week + substances) - Isolation Forest',
    cm_out / 'cm_multiclass_if_normalair_week.png'
)
cm_ae = _plot_multiclass_cm(
    y_test, y_pred_ae, classes2,
    'Multiclass CM (NormalAir_week + substances) - Dense Autoencoder',
    cm_out / 'cm_multiclass_ae_normalair_week.png'
)

pd.DataFrame(cm_if, index=classes2, columns=classes2).to_csv(cm_out / 'cm_multiclass_if_normalair_week.csv')
pd.DataFrame(cm_ae, index=classes2, columns=classes2).to_csv(cm_out / 'cm_multiclass_ae_normalair_week.csv')

print('Saved updated multiclass confusion matrices to:', cm_out)
print('- cm_multiclass_if_normalair_week.png')
print('- cm_multiclass_ae_normalair_week.png')

print('\nIsolation Forest CM (updated):')
display(pd.DataFrame(cm_if, index=classes2, columns=classes2))
print('\nDense AE CM (updated):')
display(pd.DataFrame(cm_ae, index=classes2, columns=classes2))

Classes (updated Step 16): [np.str_('Acetone'), np.str_('NormalAir_week'), np.str_('Redidlo'), np.str_('Savo'), np.str_('Softasept'), np.str_('Vinegar')]
Softasept         488
Vinegar           434
Redidlo           405
Savo              400
Acetone           349
NormalAir_week     24
Name: count, dtype: int64
Saved updated multiclass confusion matrices to: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\multiclass_confusion_matrices
- cm_multiclass_if_normalair_week.png
- cm_multiclass_ae_normalair_week.png

Isolation Forest CM (updated):


,Acetone,NormalAir_week,Redidlo,Savo,Softasept,Vinegar
Acetone,83,4,0,0,0,0
NormalAir_week,0,4,0,1,1,0
Redidlo,0,21,63,17,0,0
Savo,0,5,0,94,0,1
Softasept,0,0,4,2,108,8
Vinegar,0,17,0,5,0,87



Dense AE CM (updated):


,Acetone,NormalAir_week,Redidlo,Savo,Softasept,Vinegar
Acetone,47,4,7,29,0,0
NormalAir_week,0,6,0,0,0,0
Redidlo,0,71,8,11,0,11
Savo,0,80,0,15,0,5
Softasept,0,13,18,34,57,0
Vinegar,0,74,0,0,0,35


> **Deprecated — do not run for submission.**  
> Use **Step 12-Final (submission pipeline)** at the end of this notebook for the unified pipeline (Steps 12–17) and final outputs under pipeline_output/final_clean_run/.

---

## Step 16 (Final Rewrite): Balanced multiclass with 100 samples per class

This rewrite enforces exactly **100 samples per class** for multiclass evaluation:

- `NormalAir_week`
- `Acetone`
- `Redidlo`
- `Softasept`
- `Savo`
- `Vinegar`

If a class has fewer than 100 samples, controlled resampling with replacement is applied.
Then confusion matrices are generated for Isolation Forest and Dense Autoencoder.

In [110]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# -------------------- Build classes --------------------
week_blocks_raw, week_days, week_names = _load_week_blocks_with_days(BASE_DIR)
if len(week_blocks_raw) == 0:
    raise ValueError("No week-normal blocks found in NormalAir folder.")

baseline_week = _compute_week_monday_baseline(week_blocks_raw, week_days)
if baseline_week is None:
    raise ValueError("No Monday blocks found in NormalAir for baseline calibration.")

# Normal class from NormalAir week folder
X_norm_week = []
y_norm_week = []
for b in week_blocks_raw:
    valid_s = get_valid_sensors(b)
    if len(valid_s) < 4:
        continue
    b_cal = normalize_block(b, baseline_week)
    feat = extract_features(b_cal, baseline_week, list(range(N_SENSORS)))  # fixed 800
    X_norm_week.append(feat)
    y_norm_week.append("NormalAir_week")

if len(X_norm_week) == 0:
    raise ValueError("No valid NormalAir_week samples after preprocessing.")

X_norm_week = np.vstack(X_norm_week)
y_norm_week = np.array(y_norm_week)

# Substance classes from substance folders
X_sub_list, y_sub_list = [], []
for sub in SUBSTANCES:
    if sub not in calibration:
        continue
    baseline = calibration[sub]
    _, _, sblks, slbls = load_substance_data(sub)
    labels = slbls if len(slbls) == len(sblks) else [f"b{i}" for i in range(len(sblks))]
    raw_blks, norm_blks, valid_meta = process_blocks(sblks, labels, baseline, sub, return_blockwise=True)
    for i in range(len(valid_meta)):
        b_cal = norm_blks[i]
        feat = extract_features(b_cal, baseline_week, list(range(N_SENSORS)))
        X_sub_list.append(feat)
        y_sub_list.append(sub)

if len(X_sub_list) == 0:
    raise ValueError("No substance samples found for multiclass setup.")

X_sub = np.vstack(X_sub_list)
y_sub = np.array(y_sub_list)

X_mc = np.vstack([X_norm_week, X_sub])
y_mc = np.concatenate([y_norm_week, y_sub])
classes = sorted(pd.unique(y_mc))

# -------------------- Balance to 100/class --------------------
TARGET_PER_CLASS = 100
rng = np.random.default_rng(42)

X_bal_parts = []
y_bal_parts = []

for c in classes:
    idx = np.where(y_mc == c)[0]
    if len(idx) == 0:
        continue
    choose = rng.choice(idx, size=TARGET_PER_CLASS, replace=(len(idx) < TARGET_PER_CLASS))
    X_bal_parts.append(X_mc[choose])
    y_bal_parts.append(np.array([c] * TARGET_PER_CLASS))

X_bal = np.vstack(X_bal_parts)
y_bal = np.concatenate(y_bal_parts)

print("Balanced class counts (exact 100 each):")
print(pd.Series(y_bal).value_counts().sort_index())

# -------------------- Split + scale --------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.25, random_state=42, stratify=y_bal
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# -------------------- One-class-per-class models --------------------
def fit_if_models(Xtr, ytr, class_names):
    models = {}
    for c in class_names:
        Xc = Xtr[ytr == c]
        if len(Xc) < 10:
            continue
        Xf, Xcal = train_test_split(Xc, test_size=0.25, random_state=42)
        mdl = IsolationForest(n_estimators=300, contamination=0.05, random_state=42)
        mdl.fit(Xf)
        sc_cal = mdl.decision_function(Xcal)
        mu = float(np.mean(sc_cal))
        sd = float(np.std(sc_cal)) if float(np.std(sc_cal)) > 1e-9 else 1.0
        models[c] = {"model": mdl, "mu": mu, "sd": sd}
    return models


def pred_if_multiclass(models, Xte, fallback_class):
    pred = []
    for i in range(len(Xte)):
        x = Xte[i:i+1]
        best_c, best_s = fallback_class, -1e18
        for c, it in models.items():
            s = float(it["model"].decision_function(x)[0])
            z = (s - it["mu"]) / it["sd"]
            if z > best_s:
                best_s, best_c = z, c
        pred.append(best_c)
    return np.array(pred)


def fit_ae_models(Xtr, ytr, class_names, hidden=(16,8,16), pca_dim=24):
    models = {}
    for c in class_names:
        Xc = Xtr[ytr == c]
        if len(Xc) < 12:
            continue
        Xf, Xcal = train_test_split(Xc, test_size=0.25, random_state=42)
        ncomp = min(pca_dim, Xf.shape[1], Xf.shape[0]-1 if Xf.shape[0] > 1 else 1)
        pca = PCA(n_components=max(1, ncomp), random_state=42)
        Zf = pca.fit_transform(Xf)
        Zcal = pca.transform(Xcal)

        ae = MLPRegressor(hidden_layer_sizes=hidden, activation="relu", solver="adam", alpha=1e-3, max_iter=800, random_state=42)
        ae.fit(Zf, Zf)

        err_cal = np.mean((Zcal - ae.predict(Zcal))**2, axis=1)
        sc_cal = -err_cal
        mu = float(np.mean(sc_cal))
        sd = float(np.std(sc_cal)) if float(np.std(sc_cal)) > 1e-9 else 1.0
        models[c] = {"pca": pca, "ae": ae, "mu": mu, "sd": sd}
    return models


def pred_ae_multiclass(models, Xte, fallback_class):
    pred = []
    for i in range(len(Xte)):
        x = Xte[i:i+1]
        best_c, best_s = fallback_class, -1e18
        for c, it in models.items():
            z = it["pca"].transform(x)
            err = float(np.mean((z - it["ae"].predict(z))**2, axis=1)[0])
            s = -err
            zsc = (s - it["mu"]) / it["sd"]
            if zsc > best_s:
                best_s, best_c = zsc, c
        pred.append(best_c)
    return np.array(pred)


def plot_multiclass_cm(y_true, y_pred, labels, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    im = ax.imshow(cm, cmap="Greens")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black", fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    return cm

fallback = classes[0]
if_models = fit_if_models(X_train_sc, y_train, classes)
ae_models = fit_ae_models(X_train_sc, y_train, classes, hidden=(16,8,16), pca_dim=24)

y_pred_if = pred_if_multiclass(if_models, X_test_sc, fallback)
y_pred_ae = pred_ae_multiclass(ae_models, X_test_sc, fallback)

out_dir = OUTPUT_DIR / "final_report_assets" / "multiclass_confusion_matrices"
out_dir.mkdir(parents=True, exist_ok=True)

cm_if = plot_multiclass_cm(y_test, y_pred_if, classes, "Multiclass CM (100/class) - Isolation Forest", out_dir / "cm_multiclass_if_100_per_class.png")
cm_ae = plot_multiclass_cm(y_test, y_pred_ae, classes, "Multiclass CM (100/class) - Dense Autoencoder", out_dir / "cm_multiclass_ae_100_per_class.png")

pd.DataFrame(cm_if, index=classes, columns=classes).to_csv(out_dir / "cm_multiclass_if_100_per_class.csv")
pd.DataFrame(cm_ae, index=classes, columns=classes).to_csv(out_dir / "cm_multiclass_ae_100_per_class.csv")

print("Saved balanced (100/class) multiclass outputs to:", out_dir)
print("- cm_multiclass_if_100_per_class.png")
print("- cm_multiclass_ae_100_per_class.png")
print("\nIsolation Forest CM:")
display(pd.DataFrame(cm_if, index=classes, columns=classes))
print("\nDense AE CM:")
display(pd.DataFrame(cm_ae, index=classes, columns=classes))

Balanced class counts (exact 100 each):
Acetone           100
NormalAir_week    100
Redidlo           100
Savo              100
Softasept         100
Vinegar           100
Name: count, dtype: int64
Saved balanced (100/class) multiclass outputs to: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\multiclass_confusion_matrices
- cm_multiclass_if_100_per_class.png
- cm_multiclass_ae_100_per_class.png

Isolation Forest CM:


,Acetone,NormalAir_week,Redidlo,Savo,Softasept,Vinegar
Acetone,14,0,0,0,8,3
NormalAir_week,0,19,0,6,0,0
Redidlo,0,8,14,3,0,0
Savo,0,3,3,17,0,2
Softasept,0,1,0,0,24,0
Vinegar,0,5,0,4,0,16



Dense AE CM:


,Acetone,NormalAir_week,Redidlo,Savo,Softasept,Vinegar
Acetone,8,0,17,0,0,0
NormalAir_week,0,17,8,0,0,0
Redidlo,0,2,19,1,1,2
Savo,0,1,17,7,0,0
Softasept,0,0,23,0,2,0
Vinegar,0,1,13,1,0,10


> **Deprecated — do not run for submission.**  
> Use **Step 12-Final (submission pipeline)** at the end of this notebook for the unified pipeline (Steps 12–17) and final outputs under pipeline_output/final_clean_run/.

---

## Step 17: Confidence-aware multiclass classification (score table + reject class)

This step adds clear class decision logic:

1. Compute **per-class scores** for every test sample.
2. Compute `top1`, `top2`, and `margin = top1 - top2`.
3. Apply reject rule:
   - if `top1_score < score_threshold` OR `margin < margin_threshold` -> `Unknown`
4. Save score table and evaluate:
   - confusion matrix with `Unknown`
   - reject rate
   - per-class precision/recall/F1 (including support)

Outputs are saved to `pipeline_output/final_report_assets/step17_confidence/`.

In [111]:
from sklearn.metrics import classification_report, confusion_matrix

# Reuse balanced 100/class split from Step 16 final rewrite.
# Expected variables from Step 16 final rewrite:
# classes, X_train_sc, X_test_sc, y_test
# + model-fit helpers fit_if_models, fit_ae_models

if 'classes' not in globals() or 'X_train_sc' not in globals() or 'X_test_sc' not in globals() or 'y_test' not in globals():
    raise RuntimeError("Run Step 16 (Final Rewrite) first.")

out17 = OUTPUT_DIR / "final_report_assets" / "step17_confidence"
out17.mkdir(parents=True, exist_ok=True)


def _if_score_matrix(Xtr, ytr, Xte, class_names):
    models = fit_if_models(Xtr, ytr, class_names)
    score_mat = np.zeros((len(Xte), len(class_names)), dtype=float)
    for j, c in enumerate(class_names):
        if c not in models:
            score_mat[:, j] = -1e9
            continue
        mdl = models[c]["model"]
        mu = models[c]["mu"]
        sd = models[c]["sd"]
        s = mdl.decision_function(Xte)
        score_mat[:, j] = (s - mu) / sd
    return score_mat


def _ae_score_matrix(Xtr, ytr, Xte, class_names, hidden=(16,8,16), pca_dim=24):
    models = fit_ae_models(Xtr, ytr, class_names, hidden=hidden, pca_dim=pca_dim)
    score_mat = np.zeros((len(Xte), len(class_names)), dtype=float)
    for j, c in enumerate(class_names):
        if c not in models:
            score_mat[:, j] = -1e9
            continue
        pca = models[c]["pca"]
        ae = models[c]["ae"]
        mu = models[c]["mu"]
        sd = models[c]["sd"]
        z = pca.transform(Xte)
        err = np.mean((z - ae.predict(z))**2, axis=1)
        s = -err  # higher is better
        score_mat[:, j] = (s - mu) / sd
    return score_mat


def _predict_with_reject(score_mat, class_names, score_threshold=0.0, margin_threshold=0.20, reject_label="Unknown"):
    idx_sort = np.argsort(score_mat, axis=1)
    top1_idx = idx_sort[:, -1]
    top2_idx = idx_sort[:, -2]
    top1_score = score_mat[np.arange(len(score_mat)), top1_idx]
    top2_score = score_mat[np.arange(len(score_mat)), top2_idx]
    margin = top1_score - top2_score

    pred = np.array([class_names[i] for i in top1_idx], dtype=object)
    reject_mask = (top1_score < score_threshold) | (margin < margin_threshold)
    pred[reject_mask] = reject_label

    return pred, top1_score, top2_score, margin, reject_mask


def _evaluate_with_unknown(y_true, y_pred, class_names, reject_label="Unknown"):
    labels_cm = list(class_names) + [reject_label]
    cm = confusion_matrix(y_true, y_pred, labels=labels_cm)
    report = classification_report(y_true, y_pred, labels=list(class_names), output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report).T
    reject_rate = float((y_pred == reject_label).mean())
    return cm, labels_cm, report_df, reject_rate


def _plot_cm_with_unknown(cm, labels, title, save_path):
    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    im = ax.imshow(cm, cmap="Greens")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


# ---------- Isolation Forest confidence-aware ----------
score_if = _if_score_matrix(X_train_sc, y_train, X_test_sc, classes)
pred_if, t1_if, t2_if, m_if, rej_if = _predict_with_reject(
    score_if, classes, score_threshold=0.0, margin_threshold=0.20, reject_label="Unknown"
)
cm_if, labels_if, rep_if, rej_rate_if = _evaluate_with_unknown(y_test, pred_if, classes, reject_label="Unknown")

# score table
df_if = pd.DataFrame(score_if, columns=[f"score_{c}" for c in classes])
df_if.insert(0, "true_label", y_test)
df_if["pred_label"] = pred_if
df_if["top1_score"] = t1_if
df_if["top2_score"] = t2_if
df_if["margin"] = m_if
df_if["rejected"] = rej_if

# ---------- Dense AE confidence-aware ----------
score_ae = _ae_score_matrix(X_train_sc, y_train, X_test_sc, classes, hidden=(16,8,16), pca_dim=24)
pred_ae, t1_ae, t2_ae, m_ae, rej_ae = _predict_with_reject(
    score_ae, classes, score_threshold=0.0, margin_threshold=0.20, reject_label="Unknown"
)
cm_ae, labels_ae, rep_ae, rej_rate_ae = _evaluate_with_unknown(y_test, pred_ae, classes, reject_label="Unknown")

df_ae = pd.DataFrame(score_ae, columns=[f"score_{c}" for c in classes])
df_ae.insert(0, "true_label", y_test)
df_ae["pred_label"] = pred_ae
df_ae["top1_score"] = t1_ae
df_ae["top2_score"] = t2_ae
df_ae["margin"] = m_ae
df_ae["rejected"] = rej_ae

# Save outputs
(df_if).to_csv(out17 / "if_score_table_with_reject.csv", index=False)
(df_ae).to_csv(out17 / "ae_score_table_with_reject.csv", index=False)
(rep_if).to_csv(out17 / "if_classification_report.csv")
(rep_ae).to_csv(out17 / "ae_classification_report.csv")

_plot_cm_with_unknown(cm_if, labels_if, f"IF multiclass with Unknown (reject rate={rej_rate_if:.2%})", out17 / "cm_if_with_unknown.png")
_plot_cm_with_unknown(cm_ae, labels_ae, f"AE multiclass with Unknown (reject rate={rej_rate_ae:.2%})", out17 / "cm_ae_with_unknown.png")

summary17 = pd.DataFrame([
    {"model": "IsolationForest", "reject_rate": rej_rate_if, "macro_f1": rep_if.loc["macro avg", "f1-score"], "accuracy": rep_if.loc["accuracy", "precision"] if "accuracy" in rep_if.index else np.nan},
    {"model": "DenseAutoencoder", "reject_rate": rej_rate_ae, "macro_f1": rep_ae.loc["macro avg", "f1-score"], "accuracy": rep_ae.loc["accuracy", "precision"] if "accuracy" in rep_ae.index else np.nan},
])
summary17.to_csv(out17 / "step17_summary.csv", index=False)

print("Saved Step 17 outputs to:", out17)
print("- if_score_table_with_reject.csv")
print("- ae_score_table_with_reject.csv")
print("- cm_if_with_unknown.png")
print("- cm_ae_with_unknown.png")
print("- if_classification_report.csv")
print("- ae_classification_report.csv")
print("- step17_summary.csv")

print("\nStep 17 summary:")
display(summary17)
print("\nIF per-class report:")
display(rep_if)
print("\nAE per-class report:")
display(rep_ae)

Saved Step 17 outputs to: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\step17_confidence
- if_score_table_with_reject.csv
- ae_score_table_with_reject.csv
- cm_if_with_unknown.png
- cm_ae_with_unknown.png
- if_classification_report.csv
- ae_classification_report.csv
- step17_summary.csv

Step 17 summary:


,model,reject_rate,macro_f1,accuracy
0,IsolationForest,0.400000,0.614128,NaN
1,DenseAutoencoder,0.413333,0.370496,NaN



IF per-class report:


,precision,recall,f1-score,support
Acetone,1.000000,0.320000,0.484848,25.0
NormalAir_week,0.631579,0.480000,0.545455,25.0
Redidlo,1.000000,0.480000,0.648649,25.0
Savo,0.500000,0.360000,0.418605,25.0
Softasept,1.000000,0.720000,0.837209,25.0
Vinegar,1.000000,0.600000,0.750000,25.0
micro avg,0.822222,0.493333,0.616667,150.0
macro avg,0.855263,0.493333,0.614128,150.0
weighted avg,0.855263,0.493333,0.614128,150.0



AE per-class report:


,precision,recall,f1-score,support
Acetone,1.000000,0.280000,0.437500,25.0
NormalAir_week,0.850000,0.680000,0.755556,25.0
Redidlo,0.240000,0.480000,0.320000,25.0
Savo,1.000000,0.040000,0.076923,25.0
Softasept,1.000000,0.080000,0.148148,25.0
Vinegar,1.000000,0.320000,0.484848,25.0
micro avg,0.534091,0.313333,0.394958,150.0
macro avg,0.848333,0.313333,0.370496,150.0
weighted avg,0.848333,0.313333,0.370496,150.0


> **Deprecated — do not run for submission.**  
> Use **Step 12-Final (submission pipeline)** at the end of this notebook for the unified pipeline (Steps 12–17) and final outputs under pipeline_output/final_clean_run/.

---

## Step 17b: Automatic threshold tuning for confidence-aware multiclass

This extension performs grid search over:
- `score_threshold`
- `margin_threshold`

for IF and AE score matrices, and selects the best pair by:
- maximizing macro-F1
- under reject-rate constraint (default <= 20%).

Then it saves tuned confusion matrices and tuning summary.

In [112]:
from sklearn.metrics import f1_score, accuracy_score

# Requires Step 17 variables: score_if, score_ae, y_test, classes, _predict_with_reject, _evaluate_with_unknown, _plot_cm_with_unknown
required = ["score_if", "score_ae", "y_test", "classes"]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"Run Step 17 first. Missing variables: {missing}")

out17 = OUTPUT_DIR / "final_report_assets" / "step17_confidence"
out17.mkdir(parents=True, exist_ok=True)

# Tuning config
MAX_REJECT = 0.20  # <= 20%
score_grid = np.round(np.arange(-1.0, 1.01, 0.1), 2)
margin_grid = np.round(np.arange(0.0, 1.01, 0.05), 2)


def _grid_search_thresholds(score_mat, y_true, class_names, model_name, max_reject=0.2):
    rows = []
    for st in score_grid:
        for mt in margin_grid:
            pred, t1, t2, m, rej = _predict_with_reject(
                score_mat,
                class_names,
                score_threshold=float(st),
                margin_threshold=float(mt),
                reject_label="Unknown",
            )
            reject_rate = float(rej.mean())

            # Macro-F1 over known classes only (exclude Unknown label)
            f1_macro = f1_score(y_true, pred, labels=list(class_names), average="macro", zero_division=0)
            acc = accuracy_score(y_true, pred)

            # Penalized objective to avoid aggressive rejection
            objective = f1_macro - 0.2 * reject_rate

            rows.append({
                "model": model_name,
                "score_threshold": float(st),
                "margin_threshold": float(mt),
                "reject_rate": reject_rate,
                "macro_f1": float(f1_macro),
                "accuracy": float(acc),
                "objective": float(objective),
            })

    df = pd.DataFrame(rows)

    feasible = df[df["reject_rate"] <= max_reject]
    if len(feasible) > 0:
        best = feasible.sort_values(["macro_f1", "accuracy", "reject_rate"], ascending=[False, False, True]).iloc[0]
        mode = f"feasible (reject<={max_reject:.0%})"
    else:
        best = df.sort_values(["objective", "macro_f1"], ascending=[False, False]).iloc[0]
        mode = "fallback (no feasible under reject constraint)"

    return df, best, mode


# Run tuning for IF and AE
if_grid, if_best, if_mode = _grid_search_thresholds(score_if, y_test, classes, model_name="IsolationForest", max_reject=MAX_REJECT)
ae_grid, ae_best, ae_mode = _grid_search_thresholds(score_ae, y_test, classes, model_name="DenseAutoencoder", max_reject=MAX_REJECT)

# Apply best settings and produce tuned outputs
def _apply_best(score_mat, y_true, class_names, best_row, model_tag):
    pred, t1, t2, m, rej = _predict_with_reject(
        score_mat,
        class_names,
        score_threshold=float(best_row["score_threshold"]),
        margin_threshold=float(best_row["margin_threshold"]),
        reject_label="Unknown",
    )
    cm, labels_cm, rep, rr = _evaluate_with_unknown(y_true, pred, class_names, reject_label="Unknown")

    cm_path = out17 / f"cm_{model_tag}_with_unknown_tuned.png"
    _plot_cm_with_unknown(cm, labels_cm, f"{model_tag} tuned (rej={rr:.2%}, F1={best_row['macro_f1']:.3f})", cm_path)

    score_tbl = pd.DataFrame(score_mat, columns=[f"score_{c}" for c in class_names])
    score_tbl.insert(0, "true_label", y_true)
    score_tbl["pred_label"] = pred
    score_tbl["top1_score"] = t1
    score_tbl["top2_score"] = t2
    score_tbl["margin"] = m
    score_tbl["rejected"] = rej
    score_tbl["score_threshold"] = float(best_row["score_threshold"])
    score_tbl["margin_threshold"] = float(best_row["margin_threshold"])

    score_tbl.to_csv(out17 / f"{model_tag.lower()}_score_table_tuned.csv", index=False)
    rep.to_csv(out17 / f"{model_tag.lower()}_classification_report_tuned.csv")

    return {
        "model": model_tag,
        "score_threshold": float(best_row["score_threshold"]),
        "margin_threshold": float(best_row["margin_threshold"]),
        "reject_rate": float(rr),
        "macro_f1": float(best_row["macro_f1"]),
        "accuracy": float(best_row["accuracy"]),
        "mode": best_row.get("mode", ""),
        "cm_path": str(cm_path),
    }

if_best = if_best.copy(); if_best["mode"] = if_mode
ae_best = ae_best.copy(); ae_best["mode"] = ae_mode

res_if = _apply_best(score_if, y_test, classes, if_best, model_tag="IsolationForest")
res_ae = _apply_best(score_ae, y_test, classes, ae_best, model_tag="DenseAutoencoder")

# Save tuning tables
if_grid.to_csv(out17 / "if_threshold_tuning_grid.csv", index=False)
ae_grid.to_csv(out17 / "ae_threshold_tuning_grid.csv", index=False)

summary_tuned = pd.DataFrame([res_if, res_ae])
summary_tuned.to_csv(out17 / "step17_tuned_summary.csv", index=False)

print("Saved tuned outputs to:", out17)
print("- if_threshold_tuning_grid.csv")
print("- ae_threshold_tuning_grid.csv")
print("- cm_isolationforest_with_unknown_tuned.png")
print("- cm_denseautoencoder_with_unknown_tuned.png")
print("- step17_tuned_summary.csv")

print("\nBest IF setting:")
display(if_best.to_frame().T)
print("\nBest AE setting:")
display(ae_best.to_frame().T)
print("\nTuned summary:")
display(summary_tuned)

Saved tuned outputs to: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_report_assets\step17_confidence
- if_threshold_tuning_grid.csv
- ae_threshold_tuning_grid.csv
- cm_isolationforest_with_unknown_tuned.png
- cm_denseautoencoder_with_unknown_tuned.png
- step17_tuned_summary.csv

Best IF setting:


,model,score_threshold,margin_threshold,reject_rate,macro_f1,accuracy,objective,mode
4,IsolationForest,-1.0,0.2,0.186667,0.675982,0.613333,0.638649,feasible (reject<=20%)



Best AE setting:


,model,score_threshold,margin_threshold,reject_rate,macro_f1,accuracy,objective,mode
0,DenseAutoencoder,-1.0,0.0,0.086667,0.42126,0.38,0.403926,feasible (reject<=20%)



Tuned summary:


,model,score_threshold,margin_threshold,reject_rate,macro_f1,accuracy,mode,cm_path
0,IsolationForest,-1.0,0.2,0.186667,0.675982,0.613333,feasible (reject<=20%),C:\Users\yonat\OneDrive\Desktop\Nw File Anomal...
1,DenseAutoencoder,-1.0,0.0,0.086667,0.421260,0.380000,feasible (reject<=20%),C:\Users\yonat\OneDrive\Desktop\Nw File Anomal...


## Step 12-Final (submission pipeline)

Run the **two code cells** below in order (helpers, then run). Requires earlier cells through calibration and substance loading.

**Output folder:** `pipeline_output/final_clean_run/` (`tables/`, `cm/`).

**Pipeline:** NormalAir → Monday baseline → manual (800) + 1D-conv features → binary IF/AE with 1000/1000 samples → multiclass 100/class → Unknown reject + threshold grid search.


In [113]:
# Step 12-Final — helpers (run after Steps 1–11)
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor

OUT = OUTPUT_DIR / "final_clean_run"
(OUT / "tables").mkdir(parents=True, exist_ok=True)
(OUT / "cm").mkdir(parents=True, exist_ok=True)

_NORMALAIR_NAMES = (
    "NormalAir", "Normal Air Week", "Normal_Air_Week", "Week Normal Air", "Air Week", "Air New",
)


def _normalair_folder(base: Path) -> Path | None:
    for name in _NORMALAIR_NAMES:
        p = base / name
        if p.is_dir():
            return p
    return None


def _weekday_from_path(path) -> str:
    s = str(path).lower()
    keys = (
        ("mon", "Monday"), ("tue", "Tuesday"), ("wed", "Wednesday"),
        ("thu", "Thursday"), ("fri", "Friday"), ("sat", "Saturday"), ("sun", "Sunday"),
    )
    for k, day in keys:
        if k in s:
            return day
    return "Unknown"


def export_week_long_csv(base_dir: Path, out_csv: Path) -> pd.DataFrame:
    folder = _normalair_folder(base_dir)
    if folder is None:
        raise FileNotFoundError("NormalAir folder not found under BASE_DIR.")
    rows = []
    files = sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in {".bmerawdata", ".csv"}
    )
    for f in files:
        day = _weekday_from_path(f)
        for bi, blk in enumerate(load_blocks_from_file(f)):
            for (s, st), vals in blk.items():
                for i, g in enumerate(vals):
                    rows.append(
                        {"file": f.name, "day": day, "block": bi, "sensor": s, "step": st, "i": i, "gr": float(g)}
                    )
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df


def conv1d_features(block, valid_sensors=None):
    kernels = (
        np.array([1.0, -1.0]),
        np.array([1.0, -2.0, 1.0]),
        np.array([1.0, 0.0, -1.0]),
    )
    if valid_sensors is None:
        valid_sensors = list(range(N_SENSORS))
    feat = []
    for s in range(N_SENSORS):
        for st in range(N_STEPS):
            if s not in valid_sensors:
                feat.extend([0.0, 0.0, 0.0])
                continue
            arr = np.asarray(block.get((s, st), []), dtype=float)
            if arr.size < 3:
                feat.extend([0.0, 0.0, 0.0])
                continue
            for k in kernels:
                feat.append(float(np.mean(np.abs(np.convolve(arr, k, mode="valid")))))
    return np.asarray(feat, dtype=float)


def k_of_m_vote(pred: np.ndarray, k: int, m: int) -> np.ndarray:
    pred = np.asarray(pred, dtype=int)
    out = np.zeros_like(pred)
    for i in range(len(pred)):
        s0 = max(0, i - m + 1)
        out[i] = int(pred[s0 : i + 1].sum() >= k)
    return out


def if_train_binary(X_train, X_norm, X_anom, fpr_cap=0.25):
    X_fit, X_cal = train_test_split(X_train, test_size=0.25, random_state=42)
    mdl = IsolationForest(n_estimators=300, contamination=0.05, random_state=42)
    mdl.fit(X_fit)
    sc_cal = mdl.decision_function(X_cal)
    sc_n = mdl.decision_function(X_norm)
    sc_a = mdl.decision_function(X_anom)
    rows = []
    for pct in range(1, 41):
        thr = np.percentile(sc_cal, pct)
        rows.append((pct, float(thr), float((sc_n < thr).mean()), float((sc_a < thr).mean())))
    df = pd.DataFrame(rows, columns=["pct", "thr", "fpr", "tpr"])
    cand = df[df.fpr <= fpr_cap]
    if len(cand):
        best = cand.sort_values(["tpr", "fpr"], ascending=[False, True]).iloc[0]
    else:
        best = df.assign(score=df.tpr - df.fpr).sort_values("score", ascending=False).iloc[0]
    p_n = (sc_n < best.thr).astype(int)
    p_a = (sc_a < best.thr).astype(int)
    return p_n, p_a, df, int(best.pct), float(best.thr)


def ae_train_binary(X_train, X_norm, X_anom, hidden=(16, 8, 16), pca_dim=24, fpr_cap=0.25):
    X_fit, X_cal = train_test_split(X_train, test_size=0.25, random_state=42)
    ncomp = min(pca_dim, X_fit.shape[1], max(1, X_fit.shape[0] - 1))
    pca = PCA(n_components=max(1, ncomp), random_state=42)
    Z_fit = pca.fit_transform(X_fit)
    Z_cal = pca.transform(X_cal)
    Z_n = pca.transform(X_norm)
    Z_a = pca.transform(X_anom)
    ae = MLPRegressor(
        hidden_layer_sizes=hidden, activation="relu", solver="adam",
        alpha=1e-3, max_iter=800, random_state=42,
    )
    ae.fit(Z_fit, Z_fit)
    e_cal = np.mean((Z_cal - ae.predict(Z_cal)) ** 2, axis=1)
    e_n = np.mean((Z_n - ae.predict(Z_n)) ** 2, axis=1)
    e_a = np.mean((Z_a - ae.predict(Z_a)) ** 2, axis=1)
    rows = []
    for pct in range(60, 100):
        thr = np.percentile(e_cal, pct)
        rows.append((pct, float(thr), float((e_n >= thr).mean()), float((e_a >= thr).mean())))
    df = pd.DataFrame(rows, columns=["pct", "thr", "fpr", "tpr"])
    cand = df[df.fpr <= fpr_cap]
    if len(cand):
        best = cand.sort_values(["tpr", "fpr"], ascending=[False, True]).iloc[0]
    else:
        best = df.assign(score=df.tpr - df.fpr).sort_values("score", ascending=False).iloc[0]
    p_n = (e_n >= best.thr).astype(int)
    p_a = (e_a >= best.thr).astype(int)
    return p_n, p_a, df, int(best.pct), float(best.thr), pca, ae


def binary_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return {
        "accuracy": float((y_true == y_pred).mean()),
        "fpr": float(fp / max(tn + fp, 1)),
        "recall": float(rec),
        "precision": float(prec),
        "f1": float(f1),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def plot_cm(cm, labels, title, out_path, cmap="Greens"):
    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    im = ax.imshow(cm, cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()


def best_k_of_m(p_n, p_a, score_fn):
    best = None
    for m in (3, 5, 7):
        for k in range(1, m + 1):
            p = np.concatenate([k_of_m_vote(p_n, k, m), k_of_m_vote(p_a, k, m)])
            y = np.concatenate([np.zeros(1000, dtype=int), np.ones(1000, dtype=int)])
            met = binary_metrics(y, p)
            score = score_fn(met)
            if best is None or score > best["score"]:
                best = {"k": k, "m": m, "pred": p, "metrics": met, "score": score}
    return best


def run_binary_eval(X_week_feat, X_sub_feat, week_days, tag: str):
    tr = np.array([d in {"Monday", "Tuesday", "Wednesday"} for d in week_days])
    te = np.array([d in {"Thursday", "Friday"} for d in week_days])
    sc = StandardScaler()
    Xtr = sc.fit_transform(X_week_feat[tr])
    Xn = sc.transform(X_week_feat[te])
    Xa = sc.transform(X_sub_feat)
    rng = np.random.default_rng(42)
    Xn = Xn[rng.choice(len(Xn), size=1000, replace=(len(Xn) < 1000))]
    Xa = Xa[rng.choice(len(Xa), size=1000, replace=(len(Xa) < 1000))]

    pn_if, pa_if, sweep_if, pct_if, thr_if = if_train_binary(Xtr, Xn, Xa)
    best_if = best_k_of_m(pn_if, pa_if, lambda m: m["recall"] - 0.5 * m["fpr"])

    pn_ae, pa_ae, sweep_ae, pct_ae, thr_ae, _, _ = ae_train_binary(Xtr, Xn, Xa)
    best_ae = best_k_of_m(pn_ae, pa_ae, lambda m: m["recall"] - 0.5 * m["fpr"])

    y_true = np.concatenate([np.zeros(1000, dtype=int), np.ones(1000, dtype=int)])
    cm_if = confusion_matrix(y_true, best_if["pred"], labels=[0, 1])
    cm_ae = confusion_matrix(y_true, best_ae["pred"], labels=[0, 1])
    plot_cm(cm_if, ["Normal", "Anomaly"], f"Binary IF ({tag}) 1000/1000", OUT / "cm" / f"cm_binary_if_{tag}.png")
    plot_cm(cm_ae, ["Normal", "Anomaly"], f"Binary AE ({tag}) 1000/1000", OUT / "cm" / f"cm_binary_ae_{tag}.png")
    sweep_if.to_csv(OUT / "tables" / f"sweep_if_{tag}.csv", index=False)
    sweep_ae.to_csv(OUT / "tables" / f"sweep_ae_{tag}.csv", index=False)

    return [
        {"branch": tag, "model": "IF", "threshold_pct": pct_if, "threshold": thr_if, "k": best_if["k"], "m": best_if["m"], **best_if["metrics"]},
        {"branch": tag, "model": "AE", "threshold_pct": pct_ae, "threshold": thr_ae, "k": best_ae["k"], "m": best_ae["m"], **best_ae["metrics"]},
    ]


def tune_reject_grid(S, y_true, class_names, max_reject=0.2):
    rows = []
    for st in np.round(np.arange(-1.0, 1.01, 0.1), 2):
        for mt in np.round(np.arange(0.0, 1.01, 0.05), 2):
            idx = np.argsort(S, axis=1)
            t1 = idx[:, -1]
            t2 = idx[:, -2]
            s1 = S[np.arange(len(S)), t1]
            s2 = S[np.arange(len(S)), t2]
            margin = s1 - s2
            pred = np.array([class_names[i] for i in t1], dtype=object)
            rej = (s1 < float(st)) | (margin < float(mt))
            pred[rej] = "Unknown"
            rr = float(rej.mean())
            f1m = float(f1_score(y_true, pred, labels=list(class_names), average="macro", zero_division=0))
            acc = float(accuracy_score(y_true, pred))
            rows.append({"st": float(st), "mt": float(mt), "reject_rate": rr, "macro_f1": f1m, "accuracy": acc})
    df = pd.DataFrame(rows)
    ok = df[df.reject_rate <= max_reject]
    if len(ok):
        best = ok.sort_values(["macro_f1", "accuracy", "reject_rate"], ascending=[False, False, True]).iloc[0]
    else:
        best = df.assign(obj=df.macro_f1 - 0.2 * df.reject_rate).sort_values("obj", ascending=False).iloc[0]
    return df, best


In [114]:
# Step 12-Final — run
RNG = np.random.default_rng(42)
MAX_REJECT = 0.2
SAMPLES_PER_CLASS = 100

# --- Data: week CSV + blocks + Monday baseline ---
week_df = export_week_long_csv(BASE_DIR, OUT / "tables" / "week_normal_long.csv")
wk_folder = _normalair_folder(BASE_DIR)
wk_files = sorted(
    f for f in wk_folder.rglob("*") if f.is_file() and f.suffix.lower() in {".bmerawdata", ".csv"}
)
week_blocks, week_days = [], []
for f in wk_files:
    d = _weekday_from_path(f)
    for b in load_blocks_from_file(f):
        wb = apply_windowing(b, n=N_WARMUP, m=M_STEADY)
        if len(get_valid_sensors(wb)) >= 4:
            week_blocks.append(wb)
            week_days.append(d)
week_days = np.array(week_days)
mon_blocks = [b for b, d in zip(week_blocks, week_days) if d == "Monday"]
if not mon_blocks:
    raise ValueError("No Monday blocks for week baseline.")
baseline_week = compute_baseline(mon_blocks, apply_windowing_flag=False)

# --- Features ---
X_week_manual = np.vstack([
    extract_features(normalize_block(b, baseline_week), baseline_week, list(range(N_SENSORS)))
    for b in week_blocks
])
X_week_auto = np.vstack([
    conv1d_features(normalize_block(b, baseline_week), valid_sensors=get_valid_sensors(b))
    for b in week_blocks
])

X_sub_manual, X_sub_auto, y_sub = [], [], []
for sub in SUBSTANCES:
    if sub not in calibration:
        continue
    baseline = calibration[sub]
    _, _, sblks, slbls = load_substance_data(sub)
    labels = slbls if len(slbls) == len(sblks) else [f"b{i}" for i in range(len(sblks))]
    _, norm_blks, meta = process_blocks(sblks, labels, baseline, sub, return_blockwise=True)
    for i in range(len(meta)):
        nb = norm_blks[i]
        X_sub_manual.append(extract_features(nb, baseline_week, list(range(N_SENSORS))))
        X_sub_auto.append(conv1d_features(nb, valid_sensors=list(range(N_SENSORS))))
        y_sub.append(sub)
X_sub_manual = np.vstack(X_sub_manual)
X_sub_auto = np.vstack(X_sub_auto)
y_sub = np.array(y_sub)

# --- Binary anomaly (1000 normal + 1000 anomaly) ---
rows_bin = []
rows_bin += run_binary_eval(X_week_manual, X_sub_manual, week_days, "manual_stats_800")
rows_bin += run_binary_eval(X_week_auto, X_sub_auto, week_days, "auto_conv_240")
binary_summary = pd.DataFrame(rows_bin)
binary_summary.to_csv(OUT / "tables" / "binary_summary_final.csv", index=False)
display(binary_summary)

# --- Multiclass: 100 samples per class ---
X_norm_mc = X_week_manual
y_norm_mc = np.array(["NormalAir_week"] * len(X_norm_mc))
X_all = np.vstack([X_norm_mc, X_sub_manual])
y_all = np.concatenate([y_norm_mc, y_sub])
classes_mc = sorted(pd.unique(y_all))

Xs, ys = [], []
for c in classes_mc:
    idx = np.where(y_all == c)[0]
    pick = RNG.choice(idx, size=SAMPLES_PER_CLASS, replace=(len(idx) < SAMPLES_PER_CLASS))
    Xs.append(X_all[pick])
    ys.append(np.array([c] * SAMPLES_PER_CLASS))
Xb = np.vstack(Xs)
yb = np.concatenate(ys)

Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.25, random_state=42, stratify=yb)
scm = StandardScaler()
Xtrs = scm.fit_transform(Xtr)
Xtes = scm.transform(Xte)

if_models = {}
for c in classes_mc:
    Xc = Xtrs[ytr == c]
    xf, xcal = train_test_split(Xc, test_size=0.25, random_state=42)
    m = IsolationForest(n_estimators=300, contamination=0.05, random_state=42)
    m.fit(xf)
    sc = m.decision_function(xcal)
    sd = float(np.std(sc))
    if_models[c] = (m, float(np.mean(sc)), sd if sd > 1e-9 else 1.0)

ae_models = {}
for c in classes_mc:
    Xc = Xtrs[ytr == c]
    xf, xcal = train_test_split(Xc, test_size=0.25, random_state=42)
    ncomp = min(24, xf.shape[1], max(1, xf.shape[0] - 1))
    pca = PCA(n_components=max(1, ncomp), random_state=42)
    zf = pca.fit_transform(xf)
    zcal = pca.transform(xcal)
    ae = MLPRegressor(
        hidden_layer_sizes=(16, 8, 16), activation="relu", solver="adam",
        alpha=1e-3, max_iter=800, random_state=42,
    )
    ae.fit(zf, zf)
    err = np.mean((zcal - ae.predict(zcal)) ** 2, axis=1)
    sc = -err
    sd = float(np.std(sc))
    ae_models[c] = (pca, ae, float(np.mean(sc)), sd if sd > 1e-9 else 1.0)


def predict_if_mc(X):
    out = []
    for i in range(len(X)):
        x = X[i : i + 1]
        best_s, best_c = -1e18, classes_mc[0]
        for c, (m, mu, sd) in if_models.items():
            z = (float(m.decision_function(x)[0]) - mu) / sd
            if z > best_s:
                best_s, best_c = z, c
        out.append(best_c)
    return np.array(out)


def predict_ae_mc(X):
    out = []
    for i in range(len(X)):
        x = X[i : i + 1]
        best_s, best_c = -1e18, classes_mc[0]
        for c, (pca, ae, mu, sd) in ae_models.items():
            z = pca.transform(x)
            err = float(np.mean((z - ae.predict(z)) ** 2, axis=1)[0])
            s = (-err - mu) / sd
            if s > best_s:
                best_s, best_c = s, c
        out.append(best_c)
    return np.array(out)


yp_if = predict_if_mc(Xtes)
yp_ae = predict_ae_mc(Xtes)
cm_if_mc = confusion_matrix(yte, yp_if, labels=classes_mc)
cm_ae_mc = confusion_matrix(yte, yp_ae, labels=classes_mc)
plot_cm(cm_if_mc, classes_mc, "Multiclass IF (100/class)", OUT / "cm" / "cm_multiclass_if_100.png")
plot_cm(cm_ae_mc, classes_mc, "Multiclass AE (100/class)", OUT / "cm" / "cm_multiclass_ae_100.png")
pd.DataFrame(cm_if_mc, index=classes_mc, columns=classes_mc).to_csv(OUT / "tables" / "cm_multiclass_if_100.csv")
pd.DataFrame(cm_ae_mc, index=classes_mc, columns=classes_mc).to_csv(OUT / "tables" / "cm_multiclass_ae_100.csv")

# --- Score matrices + Unknown tuning ---
S_if = np.zeros((len(Xtes), len(classes_mc)))
S_ae = np.zeros((len(Xtes), len(classes_mc)))
for j, c in enumerate(classes_mc):
    m, mu, sd = if_models[c]
    S_if[:, j] = (m.decision_function(Xtes) - mu) / sd
    pca, ae, mu2, sd2 = ae_models[c]
    z = pca.transform(Xtes)
    e = np.mean((z - ae.predict(z)) ** 2, axis=1)
    S_ae[:, j] = ((-e) - mu2) / sd2

labels_u = classes_mc + ["Unknown"]


def apply_reject(S, st, mt):
    idx = np.argsort(S, axis=1)
    t1 = idx[:, -1]
    t2 = idx[:, -2]
    s1 = S[np.arange(len(S)), t1]
    s2 = S[np.arange(len(S)), t2]
    margin = s1 - s2
    pred = np.array([classes_mc[i] for i in t1], dtype=object)
    rej = (s1 < st) | (margin < mt)
    pred[rej] = "Unknown"
    return pred, float(rej.mean())


if_grid, if_best = tune_reject_grid(S_if, yte, classes_mc, max_reject=MAX_REJECT)
ae_grid, ae_best = tune_reject_grid(S_ae, yte, classes_mc, max_reject=MAX_REJECT)

pred_if_u, rr_if = apply_reject(S_if, float(if_best.st), float(if_best.mt))
pred_ae_u, rr_ae = apply_reject(S_ae, float(ae_best.st), float(ae_best.mt))
cm_if_u = confusion_matrix(yte, pred_if_u, labels=labels_u)
cm_ae_u = confusion_matrix(yte, pred_ae_u, labels=labels_u)
plot_cm(cm_if_u, labels_u, f"IF + Unknown (reject {rr_if:.1%})", OUT / "cm" / "cm_if_unknown_tuned.png")
plot_cm(cm_ae_u, labels_u, f"AE + Unknown (reject {rr_ae:.1%})", OUT / "cm" / "cm_ae_unknown_tuned.png")
if_grid.to_csv(OUT / "tables" / "if_tuning_grid.csv", index=False)
ae_grid.to_csv(OUT / "tables" / "ae_tuning_grid.csv", index=False)

summary = pd.DataFrame([
    {"model": "IF", "score_threshold": float(if_best.st), "margin_threshold": float(if_best.mt), "reject_rate": rr_if, "macro_f1": float(if_best.macro_f1), "accuracy": float(if_best.accuracy)},
    {"model": "AE", "score_threshold": float(ae_best.st), "margin_threshold": float(ae_best.mt), "reject_rate": rr_ae, "macro_f1": float(ae_best.macro_f1), "accuracy": float(ae_best.accuracy)},
])
summary.to_csv(OUT / "tables" / "step17_tuned_summary.csv", index=False)
display(summary)
print("Done. Outputs:", OUT.resolve())


,branch,model,threshold_pct,threshold,k,m,accuracy,fpr,recall,precision,f1,TN,FP,FN,TP
0,manual_stats_800,IF,40,0.143623,4,7,0.9625,0.050,0.975,0.951220,0.962963,950,50,25,975
1,manual_stats_800,AE,60,9.028686,3,7,0.8950,0.194,0.984,0.835314,0.903581,806,194,16,984
2,auto_conv_240,IF,40,0.069499,4,7,0.9510,0.050,0.952,0.950100,0.951049,950,50,48,952
3,auto_conv_240,AE,60,10.398142,2,7,0.9330,0.125,0.991,0.887993,0.936673,875,125,9,991


,model,score_threshold,margin_threshold,reject_rate,macro_f1,accuracy
0,IF,-1.0,0.2,0.186667,0.675982,0.613333
1,AE,-1.0,0.0,0.086667,0.421260,0.380000


Done. Outputs: C:\Users\yonat\OneDrive\Desktop\Nw File Anomaly Detection\pipeline_output\final_clean_run
